In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/deepfake_v2'

folders = [
    'roop_input_v2',
    'fake_full_v2',
    'fake_partial_v2',
    'fake_ai_generated'
]

print("=" * 70)
print("CHECKING DATASET FOLDERS")
print("=" * 70)

for folder in folders:
    path = os.path.join(BASE, folder)
    print(f"{folder:20s} -> exists={os.path.exists(path)}")

Mounted at /content/drive
CHECKING DATASET FOLDERS
roop_input_v2        -> exists=True
fake_full_v2         -> exists=True
fake_partial_v2      -> exists=True
fake_ai_generated    -> exists=True


In [ ]:
import pandas as pd
import os

MASTER_CSV = f'{BASE}/master_videos.csv'
TRAIN_CSV  = f'{BASE}/train_videos.csv'
VAL_CSV    = f'{BASE}/val_videos.csv'
TEST_CSV   = f'{BASE}/test_videos.csv'
GT_CSV     = f'{BASE}/ground_truth_partial_v2.csv'

master_df = pd.read_csv(MASTER_CSV)
train_df  = pd.read_csv(TRAIN_CSV)
val_df    = pd.read_csv(VAL_CSV)
test_df   = pd.read_csv(TEST_CSV)
gt_df     = pd.read_csv(GT_CSV)

print("Master:", master_df.shape)
print("Train :", train_df.shape)
print("Val   :", val_df.shape)
print("Test  :", test_df.shape)
print("GT    :", gt_df.shape)

print("\nFake types:")
print(master_df['fake_type'].value_counts())

print("\nLanguages:")
print(master_df['language'].value_counts())

Master: (335, 10)
Train : (234, 10)
Val   : (50, 10)
Test  : (51, 10)
GT    : (92, 9)

Fake types:
fake_type
real            92
face_swap       92
partial         92
ai_generated    59
Name: count, dtype: int64

Languages:
language
sinhala    217
tamil      118
Name: count, dtype: int64


In [ ]:
import os

def convert_to_colab_path(row):
    fake_type = str(row['fake_type']).strip().lower()
    filename = os.path.basename(
        str(row['video_path']).replace('\\', '/')
    )

    if fake_type == 'real':
        folder = 'roop_input_v2'

    elif fake_type == 'face_swap':
        folder = 'fake_full_v2'

    elif fake_type == 'partial':
        folder = 'fake_partial_v2'

    elif fake_type == 'ai_generated':
        folder = 'fake_ai_generated'

    else:
        raise ValueError(f"Unknown fake_type: {fake_type}")

    return os.path.join(
        BASE,
        folder,
        filename
    )


for df in [master_df, train_df, val_df, test_df]:
    df['colab_path'] = df.apply(
        convert_to_colab_path,
        axis=1
    )

In [ ]:
def verify_paths(name, df):
    exists = df['colab_path'].apply(os.path.exists)

    found = int(exists.sum())
    missing = int((~exists).sum())

    print("=" * 70)
    print(name)
    print("=" * 70)
    print("Total  :", len(df))
    print("Found  :", found)
    print("Missing:", missing)

    if missing > 0:
        print("\nMissing examples:")
        print(
            df.loc[
                ~exists,
                ['video_id', 'fake_type', 'colab_path']
            ].head(20).to_string(index=False)
        )

    return missing


m1 = verify_paths("MASTER", master_df)
m2 = verify_paths("TRAIN", train_df)
m3 = verify_paths("VALIDATION", val_df)
m4 = verify_paths("TEST", test_df)

MASTER
Total  : 335
Found  : 335
Missing: 0
TRAIN
Total  : 234
Found  : 234
Missing: 0
VALIDATION
Total  : 50
Found  : 50
Missing: 0
TEST
Total  : 51
Found  : 51
Missing: 0


In [ ]:
OUT_DIR = f'{BASE}/final_dataset_metadata'
os.makedirs(OUT_DIR, exist_ok=True)

master_df.to_csv(
    f'{OUT_DIR}/master_videos_colab.csv',
    index=False
)

train_df.to_csv(
    f'{OUT_DIR}/train_videos_colab.csv',
    index=False
)

val_df.to_csv(
    f'{OUT_DIR}/val_videos_colab.csv',
    index=False
)

test_df.to_csv(
    f'{OUT_DIR}/test_videos_colab.csv',
    index=False
)

print("Saved Colab-ready metadata to:")
print(OUT_DIR)

Saved Colab-ready metadata to:
/content/drive/MyDrive/deepfake_v2/final_dataset_metadata


In [ ]:
!pip -q install facenet-pytorch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 122.9 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [ ]:
!pip install -q mtcnn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.8 MB/s eta 0:00:00


In [ ]:
from mtcnn import MTCNN

detector = MTCNN()

print("MTCNN loaded successfully")

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.


MTCNN loaded successfully


In [ ]:
import cv2
import numpy as np
from mtcnn import MTCNN

detector = MTCNN()

def test_mtcnn_video(video_path, samples=30):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames <= 0:
        cap.release()
        return None

    indices = np.linspace(
        0,
        total_frames - 1,
        samples,
        dtype=int
    )

    attempted = 0
    detected = 0

    for frame_idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_idx)
        )

        ok, frame = cap.read()

        if not ok:
            continue

        attempted += 1

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        detections = detector.detect_faces(
            rgb
        )

        if detections:
            detected += 1

    cap.release()

    rate = (
        detected / attempted * 100
        if attempted > 0
        else 0
    )

    return {
        "attempted": attempted,
        "detected": detected,
        "rate": round(rate, 2)
    }

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.


In [ ]:
import cv2
from mtcnn import MTCNN

detector = MTCNN()

test_path = train_df[
    train_df["fake_type"] == "real"
].iloc[0]["colab_path"]

print("Testing:", test_path)

cap = cv2.VideoCapture(test_path)

ok, frame = cap.read()

cap.release()

print("Frame read:", ok)

if ok:
    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    detections = detector.detect_faces(
        rgb
    )

    print("Faces detected:", len(detections))

    if detections:
        print("First detection:")
        print(detections[0])

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.


Testing: /content/drive/MyDrive/deepfake_v2/roop_input_v2/tamil_010.mp4
Frame read: True
Faces detected: 0


In [ ]:
import cv2
import numpy as np

def test_mtcnn_video(
    video_path,
    samples=10
):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return {
            "error": "Could not open video"
        }

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames <= 0:

        cap.release()

        return {
            "error": "No frames"
        }

    indices = np.linspace(
        0,
        total_frames - 1,
        samples,
        dtype=int
    )

    attempted = 0
    detected = 0
    mtcnn_errors = 0

    for frame_idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_idx)
        )

        ok, frame = cap.read()

        if not ok:
            continue

        attempted += 1

        # -------------------------------------------------
        # OPTIONAL: upscale very small frames
        # -------------------------------------------------

        h, w = frame.shape[:2]

        if min(h, w) < 160:

            scale = (
                160.0 / min(h, w)
            )

            new_w = int(
                w * scale
            )

            new_h = int(
                h * scale
            )

            frame = cv2.resize(
                frame,
                (new_w, new_h),
                interpolation=cv2.INTER_CUBIC
            )

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        try:

            detections = detector.detect_faces(
                rgb
            )

            if (
                detections is not None
                and
                len(detections) > 0
            ):

                detected += 1

        except ValueError as e:

            # -------------------------------------------------
            # Some mtcnn versions crash internally when
            # there are zero candidates for ONet.
            # Treat that frame as "no face".
            # -------------------------------------------------

            if (
                "empty output" in str(e).lower()
                or
                "shape=(0" in str(e).lower()
            ):

                mtcnn_errors += 1

                continue

            raise

        except Exception as e:

            print(
                f"MTCNN warning at frame "
                f"{frame_idx}: {e}"
            )

            mtcnn_errors += 1

            continue

    cap.release()

    rate = (
        detected / attempted * 100
        if attempted > 0
        else 0
    )

    return {

        "attempted":
            attempted,

        "detected":
            detected,

        "rate":
            round(
                rate,
                2
            ),

        "mtcnn_errors":
            mtcnn_errors
    }

In [ ]:
sample_df = pd.concat([

    train_df[
        train_df["fake_type"] == "real"
    ].head(2),

    train_df[
        train_df["fake_type"] == "face_swap"
    ].head(2),

    train_df[
        train_df["fake_type"] == "partial"
    ].head(2),

    train_df[
        train_df["fake_type"] == "ai_generated"
    ].head(2),

])

for _, row in sample_df.iterrows():

    result = test_mtcnn_video(
        row["colab_path"],
        samples=10
    )

    print(
        row["video_id"],
        "|",
        row["language"],
        "|",
        row["fake_type"],
        "|",
        result
    )

tamil_010 | tamil | real | {'attempted': 10, 'detected': 2, 'rate': 20.0, 'mtcnn_errors': 5}
sinhala_042 | sinhala | real | {'attempted': 10, 'detected': 10, 'rate': 100.0, 'mtcnn_errors': 0}
sinhala_048_fake | sinhala | face_swap | {'attempted': 10, 'detected': 9, 'rate': 90.0, 'mtcnn_errors': 1}
sinhala_043_fake | sinhala | face_swap | {'attempted': 10, 'detected': 5, 'rate': 50.0, 'mtcnn_errors': 2}
tamil_011_partial | tamil | partial | {'attempted': 10, 'detected': 3, 'rate': 30.0, 'mtcnn_errors': 4}
sinhala_029_partial | sinhala | partial | {'attempted': 10, 'detected': 6, 'rate': 60.0, 'mtcnn_errors': 2}
tamil13 | tamil | ai_generated | {'attempted': 10, 'detected': 8, 'rate': 80.0, 'mtcnn_errors': 0}
sinhala21 | sinhala | ai_generated | {'attempted': 9, 'detected': 9, 'rate': 100.0, 'mtcnn_errors': 0}


In [ ]:
import cv2
import numpy as np
import pandas as pd

haar_path = (
    cv2.data.haarcascades
    + "haarcascade_frontalface_default.xml"
)

haar_detector = cv2.CascadeClassifier(
    haar_path
)

if haar_detector.empty():
    raise RuntimeError(
        "Haar Cascade could not be loaded."
    )

print("Haar detector ready.")

Haar detector ready.


In [ ]:
def mtcnn_has_face(rgb):

    try:

        detections = detector.detect_faces(
            rgb
        )

        return (
            detections is not None
            and len(detections) > 0
        )

    except ValueError as e:

        text = str(e).lower()

        if (
            "empty output" in text
            or "shape=(0" in text
        ):
            return False

        raise

    except Exception:
        return False

In [ ]:
def haar_has_face(rgb):

    gray = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2GRAY
    )

    faces = haar_detector.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=4,
        minSize=(50, 50)
    )

    return len(faces) > 0

In [ ]:
def compare_detectors_on_video(
    video_path,
    samples=20
):

    cap = cv2.VideoCapture(
        video_path
    )

    if not cap.isOpened():

        return {
            "attempted": 0,
            "haar_detected": 0,
            "mtcnn_detected": 0,
            "haar_rate": 0.0,
            "mtcnn_rate": 0.0
        }

    total_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if total_frames <= 0:

        cap.release()

        return {
            "attempted": 0,
            "haar_detected": 0,
            "mtcnn_detected": 0,
            "haar_rate": 0.0,
            "mtcnn_rate": 0.0
        }

    frame_indices = np.linspace(
        0,
        total_frames - 1,
        samples,
        dtype=int
    )

    attempted = 0
    haar_detected = 0
    mtcnn_detected = 0

    for frame_idx in frame_indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_idx)
        )

        ok, frame = cap.read()

        if not ok:
            continue

        attempted += 1

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # -----------------------
        # HAAR
        # -----------------------

        if haar_has_face(rgb):
            haar_detected += 1

        # -----------------------
        # MTCNN
        # -----------------------

        if mtcnn_has_face(rgb):
            mtcnn_detected += 1

    cap.release()

    haar_rate = (
        haar_detected / attempted * 100
        if attempted > 0
        else 0.0
    )

    mtcnn_rate = (
        mtcnn_detected / attempted * 100
        if attempted > 0
        else 0.0
    )

    return {
        "attempted":
            attempted,

        "haar_detected":
            haar_detected,

        "mtcnn_detected":
            mtcnn_detected,

        "haar_rate":
            round(haar_rate, 2),

        "mtcnn_rate":
            round(mtcnn_rate, 2)
    }

In [ ]:
def take_group(
    df,
    language,
    fake_type,
    n=4
):

    subset = df[
        (df["language"] == language)
        &
        (df["fake_type"] == fake_type)
    ]

    return subset.head(n)


benchmark_df = pd.concat([

    take_group(
        train_df,
        "sinhala",
        "real",
        4
    ),

    take_group(
        train_df,
        "tamil",
        "real",
        4
    ),

    take_group(
        train_df,
        "sinhala",
        "face_swap",
        4
    ),

    take_group(
        train_df,
        "tamil",
        "face_swap",
        4
    ),

    take_group(
        train_df,
        "sinhala",
        "partial",
        4
    ),

    take_group(
        train_df,
        "tamil",
        "partial",
        4
    ),

    take_group(
        train_df,
        "sinhala",
        "ai_generated",
        4
    ),

    take_group(
        train_df,
        "tamil",
        "ai_generated",
        4
    ),

]).reset_index(drop=True)

print(
    "Benchmark videos:",
    len(benchmark_df)
)

Benchmark videos: 32


In [ ]:
benchmark_results = []

for i, row in benchmark_df.iterrows():

    result = compare_detectors_on_video(
        row["colab_path"],
        samples=20
    )

    benchmark_results.append({
        "video_id":
            row["video_id"],

        "language":
            row["language"],

        "fake_type":
            row["fake_type"],

        **result
    })

    print(
        f"[{i+1}/{len(benchmark_df)}] "
        f"{row['video_id']} | "
        f"{row['language']} | "
        f"{row['fake_type']} | "
        f"Haar={result['haar_rate']}% | "
        f"MTCNN={result['mtcnn_rate']}%"
    )


benchmark_results_df = pd.DataFrame(
    benchmark_results
)

[1/32] sinhala_042 | sinhala | real | Haar=90.0% | MTCNN=100.0%
[2/32] sinhala_043 | sinhala | real | Haar=45.0% | MTCNN=65.0%
[3/32] sinhala_037 | sinhala | real | Haar=30.0% | MTCNN=25.0%
[4/32] sinhala_019 | sinhala | real | Haar=80.0% | MTCNN=85.0%
[5/32] tamil_010 | tamil | real | Haar=15.0% | MTCNN=45.0%
[6/32] tamil_011 | tamil | real | Haar=30.0% | MTCNN=25.0%
[7/32] tamil_023 | tamil | real | Haar=25.0% | MTCNN=65.0%
[8/32] tamil_021 | tamil | real | Haar=50.0% | MTCNN=55.0%
[9/32] sinhala_048_fake | sinhala | face_swap | Haar=40.0% | MTCNN=85.0%
[10/32] sinhala_043_fake | sinhala | face_swap | Haar=65.0% | MTCNN=65.0%
[11/32] sinhala_022_fake | sinhala | face_swap | Haar=45.0% | MTCNN=75.0%
[12/32] sinhala_034_fake | sinhala | face_swap | Haar=40.0% | MTCNN=50.0%
[13/32] tamil_026_fake | tamil | face_swap | Haar=85.0% | MTCNN=100.0%
[14/32] tamil_013_fake | tamil | face_swap | Haar=20.0% | MTCNN=45.0%
[15/32] tamil_015_fake | tamil | face_swap | Haar=25.0% | MTCNN=35.0%
[16/3

In [ ]:
print("=" * 70)
print("FINAL FACE DETECTOR COMPARISON")
print("=" * 70)

print(
    "Overall Haar:",
    round(benchmark_results_df["haar_rate"].mean(), 2),
    "%"
)

print(
    "Overall MTCNN:",
    round(benchmark_results_df["mtcnn_rate"].mean(), 2),
    "%"
)

print("\nBy language + manipulation type:")

summary = (
    benchmark_results_df
    .groupby(["language", "fake_type"])
    [["haar_rate", "mtcnn_rate"]]
    .mean()
    .round(2)
)

print(summary)

FINAL FACE DETECTOR COMPARISON
Overall Haar: 60.86 %
Overall MTCNN: 72.18 %

By language + manipulation type:
                       haar_rate  mtcnn_rate
language fake_type                          
sinhala  ai_generated      88.16       98.68
         face_swap         47.50       68.75
         partial           71.25       78.75
         real              61.25       68.75
tamil    ai_generated      91.25       96.25
         face_swap         47.50       62.50
         partial           50.00       56.25
         real              30.00       47.50


In [ ]:
benchmark_results_df.to_csv(
    "/content/drive/MyDrive/deepfake_v2/face_detector_comparison.csv",
    index=False
)

print("Saved detector comparison.")

Saved detector comparison.


In [ ]:
# ============================================================
# STEP 1 — FINAL TEMPORAL PREPROCESSING CONFIGURATION
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd
import random
from pathlib import Path

# ------------------------------------------------------------
# FINAL WINDOW DESIGN
# ------------------------------------------------------------

WINDOW_SECONDS = 2.0
WINDOW_STRIDE = 1.0

FRAMES_PER_WINDOW = 15
IMAGE_SIZE = 224

# Because MTCNN detection is not perfect on every video,
# do not require all 15 faces.
MIN_DETECTED_FACES = 8

# ------------------------------------------------------------
# TRAINING WINDOW LIMITS
# ------------------------------------------------------------

MAX_REAL_WINDOWS = 10
MAX_FULL_FAKE_WINDOWS = 10
MAX_AI_WINDOWS = 10

MAX_PARTIAL_REAL_WINDOWS = 6
MAX_PARTIAL_FAKE_WINDOWS = 6

# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

PROCESSED_ROOT = (
    f"{BASE}/final_mtcnn_windows"
)

TRAIN_OUT = (
    f"{PROCESSED_ROOT}/train"
)

VAL_OUT = (
    f"{PROCESSED_ROOT}/val"
)

TEST_OUT = (
    f"{PROCESSED_ROOT}/test"
)

for folder in [
    PROCESSED_ROOT,
    TRAIN_OUT,
    VAL_OUT,
    TEST_OUT
]:

    os.makedirs(
        folder,
        exist_ok=True
    )

print("=" * 70)
print("FINAL PREPROCESSING CONFIG")
print("=" * 70)

print("Window seconds     :", WINDOW_SECONDS)
print("Stride seconds     :", WINDOW_STRIDE)
print("Frames per window  :", FRAMES_PER_WINDOW)
print("Image size         :", IMAGE_SIZE)
print("Minimum faces      :", MIN_DETECTED_FACES)
print("Output             :", PROCESSED_ROOT)

FINAL PREPROCESSING CONFIG
Window seconds     : 2.0
Stride seconds     : 1.0
Frames per window  : 15
Image size         : 224
Minimum faces      : 8
Output             : /content/drive/MyDrive/deepfake_v2/final_mtcnn_windows


In [ ]:
# ============================================================
# STEP 2 — SAFE MTCNN FACE EXTRACTION
# ============================================================

def detect_largest_face(rgb):
    """
    Detect largest face using MTCNN.

    Returns:
        cropped RGB face, or None
    """

    try:

        detections = detector.detect_faces(
            rgb
        )

    except ValueError as e:

        text = str(e).lower()

        # Handle known empty-candidate MTCNN error
        if (
            "empty output" in text
            or
            "shape=(0" in text
        ):
            return None

        raise

    except Exception:
        return None

    if not detections:
        return None

    valid = []

    for detection in detections:

        box = detection.get(
            "box"
        )

        confidence = detection.get(
            "confidence",
            0
        )

        if not box:
            continue

        x, y, w, h = box

        if w <= 0 or h <= 0:
            continue

        # Ignore weak detections
        if confidence < 0.80:
            continue

        valid.append(
            detection
        )

    if not valid:
        return None

    # Select largest face
    best = max(
        valid,
        key=lambda d:
            d["box"][2]
            *
            d["box"][3]
    )

    x, y, w, h = best["box"]

    height, width = rgb.shape[:2]

    x = max(
        0,
        int(x)
    )

    y = max(
        0,
        int(y)
    )

    x2 = min(
        width,
        x + int(w)
    )

    y2 = min(
        height,
        y + int(h)
    )

    if (
        x2 <= x
        or
        y2 <= y
    ):
        return None

    face = rgb[
        y:y2,
        x:x2
    ]

    if face.size == 0:
        return None

    face = cv2.resize(
        face,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_AREA
    )

    return face


def extract_temporal_window(
    video_path,
    start_sec,
    end_sec
):
    """
    Extract one temporal sequence:
        shape = (15, 224, 224, 3)

    Returns:
        sequence, detected_face_count
    """

    cap = cv2.VideoCapture(
        video_path
    )

    if not cap.isOpened():
        return None, 0

    timestamps = np.linspace(
        start_sec,
        end_sec,
        FRAMES_PER_WINDOW,
        endpoint=False
    )

    detected_faces = {}

    for position, timestamp in enumerate(
        timestamps
    ):

        cap.set(
            cv2.CAP_PROP_POS_MSEC,
            float(timestamp * 1000.0)
        )

        ok, frame = cap.read()

        if not ok:
            continue

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        face = detect_largest_face(
            rgb
        )

        if face is not None:

            detected_faces[
                position
            ] = face

    cap.release()

    detected_count = len(
        detected_faces
    )

    # Reject window if too few actual face detections
    if detected_count < MIN_DETECTED_FACES:

        return None, detected_count

    # Fill occasional missing positions using
    # nearest successfully detected face
    valid_positions = sorted(
        detected_faces.keys()
    )

    sequence = []

    for position in range(
        FRAMES_PER_WINDOW
    ):

        if position in detected_faces:

            sequence.append(
                detected_faces[
                    position
                ]
            )

        else:

            nearest = min(
                valid_positions,
                key=lambda p:
                    abs(p - position)
            )

            sequence.append(
                detected_faces[
                    nearest
                ]
            )

    sequence = np.stack(
        sequence,
        axis=0
    )

    return (
        sequence.astype(
            np.uint8
        ),
        detected_count
    )


print("STEP 2 READY")

STEP 2 READY


In [ ]:
test_row = train_df.iloc[0]

sequence, face_count = extract_temporal_window(
    test_row["colab_path"],
    0.0,
    2.0
)

print("Video:", test_row["video_id"])
print("Face count:", face_count)

if sequence is None:
    print("Window rejected")
else:
    print("Sequence shape:", sequence.shape)
    print("dtype:", sequence.dtype)
    print("min:", sequence.min())
    print("max:", sequence.max())

Video: tamil_010
Face count: 0
Window rejected


In [ ]:
# ============================================================
# TEST STEP 2 ON MULTIPLE WINDOWS
# ============================================================

def test_video_windows(
    row,
    windows=None
):
    if windows is None:
        windows = [
            (0, 2),
            (10, 12),
            (20, 22),
            (30, 32),
            (40, 42)
        ]

    print("=" * 70)
    print("VIDEO:", row["video_id"])
    print("TYPE :", row["fake_type"])
    print("LANG :", row["language"])
    print("=" * 70)

    for start, end in windows:

        sequence, face_count = extract_temporal_window(
            row["colab_path"],
            start,
            end
        )

        status = (
            "ACCEPTED"
            if sequence is not None
            else "REJECTED"
        )

        print(
            f"{start:5.1f}-{end:5.1f}s"
            f" | faces={face_count:2d}/15"
            f" | {status}"
        )

In [ ]:
row_tamil = train_df[
    train_df["video_id"] == "tamil_010"
].iloc[0]

test_video_windows(
    row_tamil
)

VIDEO: tamil_010
TYPE : real
LANG : tamil
  0.0-  2.0s | faces= 0/15 | REJECTED
 10.0- 12.0s | faces= 0/15 | REJECTED
 20.0- 22.0s | faces=15/15 | ACCEPTED
 30.0- 32.0s | faces=15/15 | ACCEPTED
 40.0- 42.0s | faces= 1/15 | REJECTED


In [ ]:
row_sinhala = train_df[
    train_df["video_id"] == "sinhala_042"
].iloc[0]

test_video_windows(row_sinhala)

VIDEO: sinhala_042
TYPE : real
LANG : sinhala
  0.0-  2.0s | faces=15/15 | ACCEPTED
 10.0- 12.0s | faces= 1/15 | REJECTED
 20.0- 22.0s | faces=15/15 | ACCEPTED
 30.0- 32.0s | faces=15/15 | ACCEPTED
 40.0- 42.0s | faces=15/15 | ACCEPTED


In [ ]:
row_fake = train_df[
    train_df["fake_type"] == "face_swap"
].iloc[0]

test_video_windows(row_fake)

VIDEO: sinhala_048_fake
TYPE : face_swap
LANG : sinhala
  0.0-  2.0s | faces= 1/15 | REJECTED
 10.0- 12.0s | faces= 6/15 | REJECTED
 20.0- 22.0s | faces= 0/15 | REJECTED
 30.0- 32.0s | faces=15/15 | ACCEPTED
 40.0- 42.0s | faces=15/15 | ACCEPTED


In [ ]:
row_ai = train_df[
    train_df["fake_type"] == "ai_generated"
].iloc[0]

test_video_windows(
    row_ai,
    windows=[
        (0, 2),
        (2, 4),
        (4, 6),
        (6, 8)
    ]
)

VIDEO: tamil13
TYPE : ai_generated
LANG : tamil
  0.0-  2.0s | faces=15/15 | ACCEPTED
  2.0-  4.0s | faces=15/15 | ACCEPTED
  4.0-  6.0s | faces= 0/15 | REJECTED
  6.0-  8.0s | faces=15/15 | ACCEPTED


In [ ]:
def calculate_fake_overlap(
    window_start,
    window_end,
    fake_start,
    fake_end
):

    intersection_start = max(
        window_start,
        fake_start
    )

    intersection_end = min(
        window_end,
        fake_end
    )

    intersection = max(
        0.0,
        intersection_end - intersection_start
    )

    window_duration = (
        window_end - window_start
    )

    if window_duration <= 0:
        return 0.0

    return intersection / window_duration


def get_temporal_label(
    row,
    window_start,
    window_end
):

    fake_type = str(
        row["fake_type"]
    ).strip().lower()

    if fake_type == "real":
        return 0

    if fake_type in {
        "face_swap",
        "ai_generated"
    }:
        return 1

    if fake_type == "partial":

        fake_start = float(
            row["fake_start_sec"]
        )

        fake_end = float(
            row["fake_end_sec"]
        )

        overlap = calculate_fake_overlap(
            window_start,
            window_end,
            fake_start,
            fake_end
        )

        if overlap == 0:
            return 0

        if overlap >= 0.50:
            return 1

        return None

    raise ValueError(
        f"Unknown fake_type: {fake_type}"
    )


def create_candidate_windows(
    row
):

    duration = float(
        row["total_duration"]
    )

    if duration < WINDOW_SECONDS:
        return []

    starts = np.arange(
        0.0,
        duration - WINDOW_SECONDS + 1e-6,
        WINDOW_STRIDE
    )

    candidates = []

    for start in starts:

        end = start + WINDOW_SECONDS

        label = get_temporal_label(
            row,
            float(start),
            float(end)
        )

        if label is None:
            continue

        candidates.append({
            "start": float(start),
            "end": float(end),
            "label": int(label)
        })

    return candidates


print("STEP 3 READY")

STEP 3 READY


In [ ]:
def select_windows_for_video(
    row,
    candidates,
    split_name
):

    fake_type = str(
        row["fake_type"]
    ).lower()

    candidates = list(
        candidates
    )

    # Keep full coverage for val/test
    if split_name in {
        "val",
        "test"
    }:
        return candidates

    # TRAINING ONLY

    if fake_type == "real":

        random.shuffle(candidates)

        return candidates[
            :MAX_REAL_WINDOWS
        ]

    if fake_type == "face_swap":

        random.shuffle(candidates)

        return candidates[
            :MAX_FULL_FAKE_WINDOWS
        ]

    if fake_type == "ai_generated":

        random.shuffle(candidates)

        return candidates[
            :MAX_AI_WINDOWS
        ]

    if fake_type == "partial":

        real_candidates = [
            c
            for c in candidates
            if c["label"] == 0
        ]

        fake_candidates = [
            c
            for c in candidates
            if c["label"] == 1
        ]

        random.shuffle(
            real_candidates
        )

        random.shuffle(
            fake_candidates
        )

        selected = (
            real_candidates[
                :MAX_PARTIAL_REAL_WINDOWS
            ]
            +
            fake_candidates[
                :MAX_PARTIAL_FAKE_WINDOWS
            ]
        )

        random.shuffle(
            selected
        )

        return selected

    return []


print("STEP 4 READY")

STEP 4 READY


In [ ]:
partial_row = train_df[
    train_df["fake_type"] == "partial"
].iloc[0]

print(
    "Video:",
    partial_row["video_id"]
)

print(
    "GT fake:",
    partial_row["fake_start_sec"],
    "->",
    partial_row["fake_end_sec"]
)

candidates = create_candidate_windows(
    partial_row
)

print(
    "Total candidate windows:",
    len(candidates)
)

print(
    "REAL windows:",
    sum(
        c["label"] == 0
        for c in candidates
    )
)

print(
    "FAKE windows:",
    sum(
        c["label"] == 1
        for c in candidates
    )
)

print("\nAround fake start:")

for c in candidates:

    if (
        partial_row["fake_start_sec"] - 3
        <= c["start"]
        <= partial_row["fake_start_sec"] + 3
    ):

        print(c)

Video: tamil_011_partial
GT fake: 69.64 -> 77.92
Total candidate windows: 117
REAL windows: 109
FAKE windows: 8

Around fake start:
{'start': 67.0, 'end': 69.0, 'label': 0}
{'start': 69.0, 'end': 71.0, 'label': 1}
{'start': 70.0, 'end': 72.0, 'label': 1}
{'start': 71.0, 'end': 73.0, 'label': 1}
{'start': 72.0, 'end': 74.0, 'label': 1}


In [ ]:
print("\nAround fake end:")

for c in candidates:

    if (
        partial_row["fake_end_sec"] - 3
        <= c["start"]
        <= partial_row["fake_end_sec"] + 3
    ):
        print(c)


Around fake end:
{'start': 75.0, 'end': 77.0, 'label': 1}
{'start': 76.0, 'end': 78.0, 'label': 1}
{'start': 78.0, 'end': 80.0, 'label': 0}
{'start': 79.0, 'end': 81.0, 'label': 0}
{'start': 80.0, 'end': 82.0, 'label': 0}


In [ ]:
train_window_df, train_stats = (
    process_dataset_split(
        train_df,
        "train"
    )
)

NameError: name 'process_dataset_split' is not defined

In [ ]:
print("train_df exists:", "train_df" in globals())
print("detector exists:", "detector" in globals())
print("process_dataset_split exists:", "process_dataset_split" in globals())
print("PROCESSED_ROOT exists:", "PROCESSED_ROOT" in globals())

train_df exists: True
detector exists: True
process_dataset_split exists: False
PROCESSED_ROOT exists: True


In [ ]:
# ============================================================
# STEP 5 — PROCESS DATASET SPLIT
# ============================================================

def safe_video_filename(split_name, row_number):

    return f"{split_name}_{row_number:04d}.npz"


def process_dataset_split(df, split_name):

    if split_name == "train":
        output_dir = TRAIN_OUT

    elif split_name == "val":
        output_dir = VAL_OUT

    elif split_name == "test":
        output_dir = TEST_OUT

    else:
        raise ValueError(
            "split_name must be train, val or test"
        )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    metadata_rows = []

    stats = {
        "videos": 0,
        "videos_with_windows": 0,
        "candidate_windows": 0,
        "selected_windows": 0,
        "saved_windows": 0,
        "skipped_low_faces": 0,
        "real_windows": 0,
        "fake_windows": 0
    }

    total_videos = len(df)

    print("=" * 70)
    print(
        f"STARTING {split_name.upper()} PREPROCESSING"
    )
    print("=" * 70)

    for row_number, (_, row) in enumerate(
        df.iterrows(),
        start=1
    ):

        stats["videos"] += 1

        video_id = str(
            row["video_id"]
        )

        video_path = str(
            row["colab_path"]
        )

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if not os.path.exists(video_path):

            print(
                f"[{row_number}/{total_videos}] "
                f"{video_id} | FILE NOT FOUND"
            )

            continue

        # ----------------------------------------------------
        # Candidate temporal windows
        # ----------------------------------------------------

        candidates = create_candidate_windows(
            row
        )

        stats["candidate_windows"] += len(
            candidates
        )

        # ----------------------------------------------------
        # Select appropriate windows
        # ----------------------------------------------------

        selected = select_windows_for_video(
            row,
            candidates,
            split_name
        )

        stats["selected_windows"] += len(
            selected
        )

        X = []
        y = []

        window_starts = []
        window_ends = []
        face_counts = []

        # ----------------------------------------------------
        # Extract MTCNN temporal sequences
        # ----------------------------------------------------

        for candidate in selected:

            sequence, face_count = (
                extract_temporal_window(
                    video_path,
                    candidate["start"],
                    candidate["end"]
                )
            )

            # Not enough detected faces
            if sequence is None:

                stats[
                    "skipped_low_faces"
                ] += 1

                continue

            X.append(
                sequence
            )

            label = int(
                candidate["label"]
            )

            y.append(
                label
            )

            window_starts.append(
                candidate["start"]
            )

            window_ends.append(
                candidate["end"]
            )

            face_counts.append(
                face_count
            )

            stats[
                "saved_windows"
            ] += 1

            if label == 0:

                stats[
                    "real_windows"
                ] += 1

            else:

                stats[
                    "fake_windows"
                ] += 1

            metadata_rows.append({

                "split":
                    split_name,

                "video_id":
                    video_id,

                "language":
                    str(row["language"]),

                "fake_type":
                    str(row["fake_type"]),

                "window_start":
                    float(candidate["start"]),

                "window_end":
                    float(candidate["end"]),

                "label":
                    label,

                "face_count":
                    int(face_count)
            })

        # ----------------------------------------------------
        # SAVE THIS VIDEO
        # ----------------------------------------------------

        if len(X) > 0:

            stats[
                "videos_with_windows"
            ] += 1

            filename = safe_video_filename(
                split_name,
                row_number
            )

            save_path = os.path.join(
                output_dir,
                filename
            )

            np.savez_compressed(
                save_path,

                X=np.asarray(
                    X,
                    dtype=np.uint8
                ),

                y=np.asarray(
                    y,
                    dtype=np.int8
                ),

                starts=np.asarray(
                    window_starts,
                    dtype=np.float32
                ),

                ends=np.asarray(
                    window_ends,
                    dtype=np.float32
                ),

                face_counts=np.asarray(
                    face_counts,
                    dtype=np.int8
                )
            )

        print(
            f"[{row_number:03d}/{total_videos}] "
            f"{video_id} | "
            f"{row['language']} | "
            f"{row['fake_type']} | "
            f"selected={len(selected)} | "
            f"saved={len(X)}"
        )

    # ========================================================
    # METADATA
    # ========================================================

    metadata_df = pd.DataFrame(
        metadata_rows
    )

    metadata_path = os.path.join(
        PROCESSED_ROOT,
        f"{split_name}_window_metadata.csv"
    )

    metadata_df.to_csv(
        metadata_path,
        index=False
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    print()
    print("=" * 70)
    print(
        f"{split_name.upper()} PREPROCESSING COMPLETE"
    )
    print("=" * 70)

    for key, value in stats.items():

        print(
            f"{key:25s}: {value}"
        )

    print()
    print(
        "Metadata saved:",
        metadata_path
    )

    return metadata_df, stats


print("process_dataset_split() READY")

process_dataset_split() READY


In [ ]:
required_items = [
    "train_df",
    "detector",
    "PROCESSED_ROOT",
    "TRAIN_OUT",
    "create_candidate_windows",
    "select_windows_for_video",
    "extract_temporal_window",
    "process_dataset_split"
]

print("=" * 60)
print("PREPROCESSING READINESS CHECK")
print("=" * 60)

all_ready = True

for item in required_items:

    exists = item in globals()

    print(
        f"{item:30s}: "
        f"{'READY' if exists else 'MISSING'}"
    )

    if not exists:
        all_ready = False

print()

if all_ready:
    print("✅ EVERYTHING READY — START TRAIN PREPROCESSING")
else:
    print("❌ DO NOT START — rerun the missing cell(s)")

PREPROCESSING READINESS CHECK
train_df                      : READY
detector                      : READY
PROCESSED_ROOT                : READY
TRAIN_OUT                     : READY
create_candidate_windows      : READY
select_windows_for_video      : READY
extract_temporal_window       : READY
process_dataset_split         : READY

✅ EVERYTHING READY — START TRAIN PREPROCESSING


In [ ]:
train_window_df, train_stats = (
    process_dataset_split(
        train_df,
        "train"
    )
)

STARTING TRAIN PREPROCESSING
[001/234] tamil_010 | tamil | real | selected=10 | saved=4
[002/234] sinhala_042 | sinhala | real | selected=10 | saved=10
[003/234] tamil_011_partial | tamil | partial | selected=12 | saved=1
[004/234] tamil13 | tamil | ai_generated | selected=9 | saved=7
[005/234] sinhala_029_partial | sinhala | partial | selected=12 | saved=10
[006/234] sinhala_048_fake | sinhala | face_swap | selected=10 | saved=9
[007/234] sinhala_043_fake | sinhala | face_swap | selected=10 | saved=7
[008/234] sinhala_043 | sinhala | real | selected=10 | saved=8
[009/234] sinhala21 | sinhala | ai_generated | selected=5 | saved=5
[010/234] sinhala_037 | sinhala | real | selected=10 | saved=4
[011/234] sinhala_050_partial | sinhala | partial | selected=12 | saved=12
[012/234] tamil_026_fake | tamil | face_swap | selected=10 | saved=10
[013/234] tamil_013_fake | tamil | face_swap | selected=10 | saved=7
[014/234] tamil_015_fake | tamil | face_swap | selected=10 | saved=3
[015/234] tamil_

In [ ]:
print("=" * 70)
print("TRAIN WINDOW QA")
print("=" * 70)

print("\nLabels:")
print(
    train_window_df["label"].value_counts()
)

print("\nLanguage x Label:")
print(
    pd.crosstab(
        train_window_df["language"],
        train_window_df["label"]
    )
)

print("\nFake Type x Label:")
print(
    pd.crosstab(
        train_window_df["fake_type"],
        train_window_df["label"]
    )
)

print("\nLanguage x Fake Type:")
print(
    pd.crosstab(
        train_window_df["language"],
        train_window_df["fake_type"]
    )
)

print("\nFace count statistics:")
print(
    train_window_df["face_count"].describe()
)

print("\nMean face count by language:")
print(
    train_window_df
    .groupby("language")["face_count"]
    .mean()
    .round(2)
)

print("\nMean face count by fake type:")
print(
    train_window_df
    .groupby("fake_type")["face_count"]
    .mean()
    .round(2)
)

TRAIN WINDOW QA

Labels:
label
1    1088
0     695
Name: count, dtype: int64

Language x Label:
label       0    1
language          
sinhala   451  742
tamil     244  346

Fake Type x Label:
label           0    1
fake_type             
ai_generated    0  322
face_swap       0  470
partial       267  296
real          428    0

Language x Fake Type:
fake_type  ai_generated  face_swap  partial  real
language                                         
sinhala             240        305      372   276
tamil                82        165      191   152

Face count statistics:
count    1783.000000
mean       14.526640
std         1.405553
min         8.000000
25%        15.000000
50%        15.000000
75%        15.000000
max        15.000000
Name: face_count, dtype: float64

Mean face count by language:
language
sinhala    14.56
tamil      14.46
Name: face_count, dtype: float64

Mean face count by fake type:
fake_type
ai_generated    14.60
face_swap       14.53
partial         14.47
real     

In [ ]:
processed_ids = set(
    train_window_df["video_id"].astype(str)
)

zero_window_videos = train_df[
    ~train_df["video_id"].astype(str).isin(processed_ids)
]

print("=" * 70)
print("TRAIN VIDEOS WITH ZERO USABLE WINDOWS")
print("=" * 70)

print(
    zero_window_videos[
        [
            "video_id",
            "language",
            "fake_type",
            "total_duration"
        ]
    ].to_string(index=False)
)

print("\nTotal:", len(zero_window_videos))

TRAIN VIDEOS WITH ZERO USABLE WINDOWS
           video_id language fake_type  total_duration
     tamil_014_fake    tamil face_swap         120.000
sinhala_031_partial  sinhala   partial         120.067

Total: 2


In [ ]:
val_window_df, val_stats = process_dataset_split(
    val_df,
    "val"
)

STARTING VAL PREPROCESSING
[001/50] sinhala_039 | sinhala | real | selected=119 | saved=107
[002/50] sinhala42 | sinhala | ai_generated | selected=9 | saved=9
[003/50] tamil_003_fake | tamil | face_swap | selected=119 | saved=113
[004/50] sinhala_027 | sinhala | real | selected=119 | saved=113
[005/50] sinhala_006 | sinhala | real | selected=119 | saved=114
[006/50] sinhala_008_partial | sinhala | partial | selected=117 | saved=49
[007/50] sinhala_024_partial | sinhala | partial | selected=117 | saved=117
[008/50] tamil_025 | tamil | real | selected=87 | saved=87
[009/50] sinhala_044_fake | sinhala | face_swap | selected=119 | saved=93
[010/50] sinhala_037_partial | sinhala | partial | selected=117 | saved=26
[011/50] sinhala_005_partial | sinhala | partial | selected=117 | saved=92
[012/50] sinhala_050_fake | sinhala | face_swap | selected=77 | saved=72
[013/50] sinhala_033_partial | sinhala | partial | selected=117 | saved=58
[014/50] sinhala_023_partial | sinhala | partial | selecte

In [ ]:
print("=" * 70)
print("VALIDATION WINDOW QA")
print("=" * 70)

print("\nLabels:")
print(
    val_window_df["label"].value_counts()
)

print("\nLanguage x Label:")
print(
    pd.crosstab(
        val_window_df["language"],
        val_window_df["label"]
    )
)

print("\nFake Type x Label:")
print(
    pd.crosstab(
        val_window_df["fake_type"],
        val_window_df["label"]
    )
)

print("\nLanguage x Fake Type:")
print(
    pd.crosstab(
        val_window_df["language"],
        val_window_df["fake_type"]
    )
)

print("\nFace count statistics:")
print(
    val_window_df["face_count"].describe()
)

print("\nMean face count by language:")
print(
    val_window_df
    .groupby("language")["face_count"]
    .mean()
    .round(2)
)

print("\nMean face count by fake type:")
print(
    val_window_df
    .groupby("fake_type")["face_count"]
    .mean()
    .round(2)
)

VALIDATION WINDOW QA

Labels:
label
0    2158
1    1233
Name: count, dtype: int64

Language x Label:
label        0    1
language           
sinhala   1371  777
tamil      787  456

Fake Type x Label:
label            0     1
fake_type               
ai_generated     0    75
face_swap        0  1058
partial        906   100
real          1252     0

Language x Fake Type:
fake_type  ai_generated  face_swap  partial  real
language                                         
sinhala              48        677      537   886
tamil                27        381      469   366

Face count statistics:
count    3391.000000
mean       14.550870
std         1.368091
min         8.000000
25%        15.000000
50%        15.000000
75%        15.000000
max        15.000000
Name: face_count, dtype: float64

Mean face count by language:
language
sinhala    14.51
tamil      14.62
Name: face_count, dtype: float64

Mean face count by fake type:
fake_type
ai_generated    14.95
face_swap       14.40
partial   

In [ ]:
test_window_df, test_stats = process_dataset_split(
    test_df,
    "test"
)

STARTING TEST PREPROCESSING
[001/51] tamil_001_fake | tamil | face_swap | selected=119 | saved=109
[002/51] sinhala_008_fake | sinhala | face_swap | selected=119 | saved=53
[003/51] sinhala_009_partial | sinhala | partial | selected=117 | saved=41
[004/51] sinhala_036_fake | sinhala | face_swap | selected=119 | saved=43
[005/51] sinhala_056 | sinhala | real | selected=119 | saved=69
[006/51] tamil_009_fake | tamil | face_swap | selected=119 | saved=48
[007/51] tamil_028_partial | tamil | partial | selected=118 | saved=40
[008/51] tamil_028_fake | tamil | face_swap | selected=119 | saved=40
[009/51] sinhala_034_partial | sinhala | partial | selected=117 | saved=50
[010/51] sinhala_012_partial | sinhala | partial | selected=117 | saved=107
[011/51] sinhala24 | sinhala | ai_generated | selected=89 | saved=58
[012/51] sinhala_042_partial | sinhala | partial | selected=117 | saved=114
[013/51] tamil_002 | tamil | real | selected=119 | saved=93
[014/51] sinhala_040_fake | sinhala | face_swap

In [ ]:
print("=" * 70)
print("TEST WINDOW QA")
print("=" * 70)

print("\nLabels:")
print(
    test_window_df["label"].value_counts()
)

print("\nLanguage x Label:")
print(
    pd.crosstab(
        test_window_df["language"],
        test_window_df["label"]
    )
)

print("\nFake Type x Label:")
print(
    pd.crosstab(
        test_window_df["fake_type"],
        test_window_df["label"]
    )
)

print("\nLanguage x Fake Type:")
print(
    pd.crosstab(
        test_window_df["language"],
        test_window_df["fake_type"]
    )
)

print("\nFace count statistics:")
print(
    test_window_df["face_count"].describe()
)

print("\nMean face count by language:")
print(
    test_window_df
    .groupby("language")["face_count"]
    .mean()
    .round(2)
)

print("\nMean face count by fake type:")
print(
    test_window_df
    .groupby("fake_type")["face_count"]
    .mean()
    .round(2)
)

TEST WINDOW QA

Labels:
label
0    2114
1    1301
Name: count, dtype: int64

Language x Label:
label        0    1
language           
sinhala   1436  947
tamil      678  354

Fake Type x Label:
label            0     1
fake_type               
ai_generated     0   146
face_swap        0  1053
partial       1040   102
real          1074     0

Language x Fake Type:
fake_type  ai_generated  face_swap  partial  real
language                                         
sinhala             128        747      815   693
tamil                18        306      327   381

Face count statistics:
count    3415.000000
mean       14.576867
std         1.285033
min         8.000000
25%        15.000000
50%        15.000000
75%        15.000000
max        15.000000
Name: face_count, dtype: float64

Mean face count by language:
language
sinhala    14.64
tamil      14.44
Name: face_count, dtype: float64

Mean face count by fake type:
fake_type
ai_generated    14.30
face_swap       14.54
partial         

In [ ]:
processed_test_ids = set(
    test_window_df["video_id"].astype(str)
)

zero_test_videos = test_df[
    ~test_df["video_id"]
    .astype(str)
    .isin(processed_test_ids)
]

print("=" * 70)
print("TEST VIDEOS WITH ZERO USABLE WINDOWS")
print("=" * 70)

if len(zero_test_videos) == 0:

    print("None — all test videos have usable windows.")

else:

    print(
        zero_test_videos[
            [
                "video_id",
                "language",
                "fake_type",
                "total_duration"
            ]
        ].to_string(index=False)
    )

print("\nTotal:", len(zero_test_videos))

TEST VIDEOS WITH ZERO USABLE WINDOWS
None — all test videos have usable windows.

Total: 0


In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf

TRAIN_DIR = f"{PROCESSED_ROOT}/train"
VAL_DIR   = f"{PROCESSED_ROOT}/val"
TEST_DIR  = f"{PROCESSED_ROOT}/test"

train_files = sorted(
    glob.glob(
        os.path.join(TRAIN_DIR, "*.npz")
    )
)

val_files = sorted(
    glob.glob(
        os.path.join(VAL_DIR, "*.npz")
    )
)

test_files = sorted(
    glob.glob(
        os.path.join(TEST_DIR, "*.npz")
    )
)

print("Train files:", len(train_files))
print("Val files  :", len(val_files))
print("Test files :", len(test_files))

Train files: 232
Val files  : 50
Test files : 51


In [ ]:
def npz_window_generator(file_list):

    for file_path in file_list:

        data = np.load(
            file_path
        )

        X = data["X"]
        y = data["y"]

        for i in range(len(y)):

            x = X[i].astype(
                np.float32
            ) / 255.0

            label = np.int32(
                y[i]
            )

            yield x, label

In [ ]:
OUTPUT_SIGNATURE = (
    tf.TensorSpec(
        shape=(
            15,
            224,
            224,
            3
        ),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(),
        dtype=tf.int32
    )
)

train_ds = tf.data.Dataset.from_generator(
    lambda:
        npz_window_generator(
            train_files
        ),
    output_signature=
        OUTPUT_SIGNATURE
)

val_ds = tf.data.Dataset.from_generator(
    lambda:
        npz_window_generator(
            val_files
        ),
    output_signature=
        OUTPUT_SIGNATURE
)

test_ds = tf.data.Dataset.from_generator(
    lambda:
        npz_window_generator(
            test_files
        ),
    output_signature=
        OUTPUT_SIGNATURE
)

In [ ]:
BATCH_SIZE = 4

train_ds = (
    train_ds
    .shuffle(
        1000,
        seed=42
    )
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)

val_ds = (
    val_ds
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)

test_ds = (
    test_ds
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)

In [ ]:
from sklearn.utils.class_weight import (
    compute_class_weight
)

y_train_for_weights = (
    train_window_df[
        "label"
    ]
    .astype(int)
    .values
)

classes = np.array(
    [0, 1]
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_for_weights
)

class_weights = {
    0: float(weights[0]),
    1: float(weights[1])
}

print(class_weights)

{0: 1.2827338129496404, 1: 0.8193933823529411}


In [ ]:
from keras.saving import (
    register_keras_serializable
)

@register_keras_serializable()
class ChannelAttention(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        ratio=8,
        **kwargs
    ):
        super().__init__(
            **kwargs
        )
        self.ratio = ratio

    def build(
        self,
        input_shape
    ):

        channels = int(
            input_shape[-1]
        )

        hidden = max(
            channels // self.ratio,
            1
        )

        self.fc1 = tf.keras.layers.Dense(
            hidden,
            activation="relu"
        )

        self.fc2 = tf.keras.layers.Dense(
            channels
        )

        super().build(
            input_shape
        )

    def call(
        self,
        x
    ):

        avg = tf.reduce_mean(
            x,
            axis=[1, 2],
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=[1, 2],
            keepdims=True
        )

        attention = tf.nn.sigmoid(
            self.fc2(
                self.fc1(avg)
            )
            +
            self.fc2(
                self.fc1(mx)
            )
        )

        return (
            x * attention
        )

    def get_config(self):

        config = (
            super().get_config()
        )

        config.update({
            "ratio":
                self.ratio
        })

        return config


@register_keras_serializable()
class SpatialAttention(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        **kwargs
    ):
        super().__init__(
            **kwargs
        )

    def build(
        self,
        input_shape
    ):

        self.conv = (
            tf.keras.layers.Conv2D(
                1,
                kernel_size=7,
                padding="same",
                activation="sigmoid"
            )
        )

        super().build(
            input_shape
        )

    def call(
        self,
        x
    ):

        avg = tf.reduce_mean(
            x,
            axis=-1,
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=-1,
            keepdims=True
        )

        concat = tf.concat(
            [avg, mx],
            axis=-1
        )

        attention = (
            self.conv(
                concat
            )
        )

        return (
            x * attention
        )

In [ ]:
from tensorflow.keras.applications import (
    MobileNetV2
)

def build_condition_c_v3():

    # --------------------------------------------------
    # SPATIAL BACKBONE
    # --------------------------------------------------

    backbone = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(
            224,
            224,
            3
        )
    )

    backbone.trainable = False

    frame_input = (
        tf.keras.Input(
            shape=(
                224,
                224,
                3
            )
        )
    )

    x = backbone(
        frame_input,
        training=False
    )

    x = ChannelAttention(
        ratio=8
    )(x)

    x = SpatialAttention()(x)

    x = (
        tf.keras.layers
        .GlobalAveragePooling2D()
        (x)
    )

    x = (
        tf.keras.layers.Dense(
            256,
            activation="relu"
        )(x)
    )

    spatial_model = (
        tf.keras.Model(
            frame_input,
            x,
            name="spatial_encoder"
        )
    )

    # --------------------------------------------------
    # TEMPORAL MODEL
    # --------------------------------------------------

    sequence_input = (
        tf.keras.Input(
            shape=(
                15,
                224,
                224,
                3
            ),
            name="sequence_input"
        )
    )

    x = (
        tf.keras.layers
        .TimeDistributed(
            spatial_model,
            name="td_spatial"
        )(sequence_input)
    )

    x = (
        tf.keras.layers
        .Bidirectional(
            tf.keras.layers.GRU(
                64,
                return_sequences=True
            ),
            name="bigru_1"
        )(x)
    )

    x = (
        tf.keras.layers
        .Bidirectional(
            tf.keras.layers.GRU(
                32
            ),
            name="bigru_2"
        )(x)
    )

    x = tf.keras.layers.Dense(
        64,
        activation="relu"
    )(x)

    x = tf.keras.layers.Dropout(
        0.4
    )(x)

    output = tf.keras.layers.Dense(
        2,
        activation="softmax",
        name="output"
    )(x)

    model = tf.keras.Model(
        sequence_input,
        output,
        name="Condition_C_v3"
    )

    return model, backbone

In [ ]:
model, backbone = (
    build_condition_c_v3()
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "Condition_C_v3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequence_input (InputLayer)     │ (None, 15, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_spatial (TimeDistributed)    │ (None, 15, 256)        │     2,997,059 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bigru_1 (Bidirectional)         │ (None, 15, 128)        │       123,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bigru_2 (Bidirectional)         │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,156,101 (12.04 MB)

 Trainable params: 898,117 (3.43 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=
        "sparse_categorical_crossentropy",
    metrics=[
        "accuracy"
    ]
)

In [ ]:
MODEL_DIR = (
    f"{BASE}/deepfake_models"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

PHASE1_BEST = (
    f"{MODEL_DIR}/"
    "condition_c_v3_phase1_best.keras"
)

callbacks_phase1 = [

    tf.keras.callbacks.ModelCheckpoint(
        PHASE1_BEST,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
history1 = model.fit(
    train_ds,
    validation_data=
        val_ds,
    epochs=15,
    class_weight=
        class_weights,
    callbacks=
        callbacks_phase1
)

Epoch 1/15
    446/Unknown 160s 170ms/step - accuracy: 0.5813 - loss: 0.6352

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: val_loss improved from None to 0.39862, saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_phase1_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_phase1_best.keras
446/446 ━━━━━━━━━━━━━━━━━━━━ 312s 510ms/step - accuracy: 0.6697 - loss: 0.5691 - val_accuracy: 0.8384 - val_loss: 0.3986 - learning_rate: 1.0000e-04
Epoch 2/15
446/446 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.8278 - loss: 0.3931
Epoch 2: val_loss improved from 0.39862 to 0.29577, saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_phase1_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_phase1_best.keras
446/446 ━━━━━━━━━━━━━━━━━━━━ 226s 466ms/step - accuracy: 0.8357 - loss: 0.3744 - val_accuracy: 0.8935 - val_loss: 0.2958 - learning_rate: 1.0000e-04
Epoch 3/15
446/446 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.8617 -

In [ ]:
# ============================================================
# PHASE 1 — VALIDATION EVALUATION
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# TRUE VALIDATION LABELS
# ------------------------------------------------------------

y_val_true = (
    val_window_df["label"]
    .astype(int)
    .values
)

# ------------------------------------------------------------
# PREDICTIONS
# ------------------------------------------------------------

val_probs = model.predict(
    val_ds,
    verbose=1
)

# Probability of FAKE class
val_fake_probs = val_probs[:, 1]

# Temporary default threshold
val_pred = (
    val_fake_probs >= 0.50
).astype(int)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

val_accuracy = accuracy_score(
    y_val_true,
    val_pred
)

val_precision = precision_score(
    y_val_true,
    val_pred,
    zero_division=0
)

val_recall = recall_score(
    y_val_true,
    val_pred,
    zero_division=0
)

val_f1 = f1_score(
    y_val_true,
    val_pred,
    zero_division=0
)

val_auc = roc_auc_score(
    y_val_true,
    val_fake_probs
)

cm = confusion_matrix(
    y_val_true,
    val_pred
)

print("=" * 70)
print("CONDITION C v3 — PHASE 1 VALIDATION")
print("=" * 70)

print(f"Accuracy  : {val_accuracy:.4f}")
print(f"Precision : {val_precision:.4f}")
print(f"Recall    : {val_recall:.4f}")
print(f"F1-score  : {val_f1:.4f}")
print(f"ROC-AUC   : {val_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_val_true,
        val_pred,
        target_names=[
            "REAL",
            "FAKE"
        ],
        digits=4,
        zero_division=0
    )
)

848/848 ━━━━━━━━━━━━━━━━━━━━ 164s 172ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


CONDITION C v3 — PHASE 1 VALIDATION
Accuracy  : 0.9148
Precision : 0.8882
Recall    : 0.8759
F1-score  : 0.8820
ROC-AUC   : 0.9684

Confusion Matrix:
[[2022  136]
 [ 153 1080]]

Classification Report:
              precision    recall  f1-score   support

        REAL     0.9297    0.9370    0.9333      2158
        FAKE     0.8882    0.8759    0.8820      1233

    accuracy                         0.9148      3391
   macro avg     0.9089    0.9064    0.9076      3391
weighted avg     0.9146    0.9148    0.9146      3391



In [ ]:
import pandas as pd
import os

HISTORY_DIR = (
    f"{BASE}/final_training_results"
)

os.makedirs(
    HISTORY_DIR,
    exist_ok=True
)

history1_df = pd.DataFrame(
    history1.history
)

history1_df.to_csv(
    f"{HISTORY_DIR}/"
    "condition_c_v3_phase1_history.csv",
    index=False
)

print(
    "Saved Phase 1 history."
)

Saved Phase 1 history.


In [ ]:
# ============================================================
# PHASE 2 — CONTROLLED MOBILE NET FINE-TUNING
# ============================================================

backbone.trainable = True

# First freeze everything
for layer in backbone.layers:
    layer.trainable = False

# Unfreeze only final 20 layers
for layer in backbone.layers[-20:]:

    # Keep BatchNorm frozen.
    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True


trainable_backbone = sum(
    layer.trainable
    for layer in backbone.layers
)

print(
    "Trainable MobileNetV2 layers:",
    trainable_backbone
)

Trainable MobileNetV2 layers: 13


In [ ]:
PHASE2_LR = 5e-6

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=PHASE2_LR
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

NameError: name 'model' is not defined

In [ ]:
import os
import glob
import math
import numpy as np
import pandas as pd
import tensorflow as tf

BASE = "/content/drive/MyDrive/deepfake_v2"

PROCESSED_ROOT = f"{BASE}/final_mtcnn_windows"

TRAIN_DIR = f"{PROCESSED_ROOT}/train"
VAL_DIR   = f"{PROCESSED_ROOT}/val"
TEST_DIR  = f"{PROCESSED_ROOT}/test"

MODEL_DIR = f"{BASE}/deepfake_models"

PHASE1_BEST = (
    f"{MODEL_DIR}/condition_c_v3_phase1_best.keras"
)

print("Phase 1 model exists:", os.path.exists(PHASE1_BEST))

Phase 1 model exists: True


In [ ]:
from keras.saving import register_keras_serializable


@register_keras_serializable()
class ChannelAttention(tf.keras.layers.Layer):

    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):

        channels = int(input_shape[-1])

        hidden = max(
            channels // self.ratio,
            1
        )

        self.fc1 = tf.keras.layers.Dense(
            hidden,
            activation="relu"
        )

        self.fc2 = tf.keras.layers.Dense(
            channels
        )

        super().build(input_shape)

    def call(self, x):

        avg = tf.reduce_mean(
            x,
            axis=[1, 2],
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=[1, 2],
            keepdims=True
        )

        attention = tf.nn.sigmoid(
            self.fc2(self.fc1(avg))
            +
            self.fc2(self.fc1(mx))
        )

        return x * attention

    def get_config(self):

        config = super().get_config()

        config.update({
            "ratio": self.ratio
        })

        return config


@register_keras_serializable()
class SpatialAttention(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):

        self.conv = tf.keras.layers.Conv2D(
            1,
            kernel_size=7,
            padding="same",
            activation="sigmoid"
        )

        super().build(input_shape)

    def call(self, x):

        avg = tf.reduce_mean(
            x,
            axis=-1,
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=-1,
            keepdims=True
        )

        concat = tf.concat(
            [avg, mx],
            axis=-1
        )

        return x * self.conv(concat)

In [ ]:
model = tf.keras.models.load_model(
    PHASE1_BEST,
    custom_objects={
        "ChannelAttention": ChannelAttention,
        "SpatialAttention": SpatialAttention
    },
    compile=False,
    safe_mode=False
)

print("MODEL RELOADED")

print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

MODEL RELOADED
Input shape : (None, 15, 224, 224, 3)
Output shape: (None, 2)


In [ ]:
train_window_df = pd.read_csv(
    f"{PROCESSED_ROOT}/train_window_metadata.csv"
)

val_window_df = pd.read_csv(
    f"{PROCESSED_ROOT}/val_window_metadata.csv"
)

test_window_df = pd.read_csv(
    f"{PROCESSED_ROOT}/test_window_metadata.csv"
)

print(
    len(train_window_df),
    len(val_window_df),
    len(test_window_df)
)

1783 3391 3415


In [ ]:
train_files = sorted(
    glob.glob(
        f"{TRAIN_DIR}/*.npz"
    )
)

val_files = sorted(
    glob.glob(
        f"{VAL_DIR}/*.npz"
    )
)

test_files = sorted(
    glob.glob(
        f"{TEST_DIR}/*.npz"
    )
)

print("Train files:", len(train_files))
print("Val files  :", len(val_files))
print("Test files :", len(test_files))

Train files: 232
Val files  : 50
Test files : 51


In [ ]:
def npz_window_generator(file_list):

    for file_path in file_list:

        data = np.load(
            file_path
        )

        X = data["X"]
        y = data["y"]

        for i in range(len(y)):

            yield (
                X[i].astype(np.float32) / 255.0,
                np.int32(y[i])
            )

In [ ]:
OUTPUT_SIGNATURE = (
    tf.TensorSpec(
        shape=(15, 224, 224, 3),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(),
        dtype=tf.int32
    )
)

BATCH_SIZE = 4

train_ds = tf.data.Dataset.from_generator(
    lambda: npz_window_generator(train_files),
    output_signature=OUTPUT_SIGNATURE
)

val_ds = tf.data.Dataset.from_generator(
    lambda: npz_window_generator(val_files),
    output_signature=OUTPUT_SIGNATURE
)

train_ds = (
    train_ds
    .shuffle(
        1000,
        seed=42
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_window_df["label"].astype(int).values
)

class_weights = {
    0: float(weights[0]),
    1: float(weights[1])
}

print("Class weights:", class_weights)

Class weights: {0: 1.2827338129496404, 1: 0.8193933823529411}


In [ ]:
for i, layer in enumerate(model.layers):
    print(
        i,
        layer.name,
        type(layer).__name__
    )

0 sequence_input InputLayer
1 td_spatial TimeDistributed
2 bigru_1 Bidirectional
3 bigru_2 Bidirectional
4 dense_3 Dense
5 dropout Dropout
6 output Dense


In [ ]:
td_layer = model.get_layer(
    "td_spatial"
)

spatial_model = td_layer.layer

print(
    "Spatial model:",
    spatial_model.name
)

for i, layer in enumerate(
    spatial_model.layers
):
    print(
        i,
        layer.name,
        type(layer).__name__
    )

Spatial model: spatial_encoder
0 input_layer_1 InputLayer
1 mobilenetv2_1.00_224 Functional
2 channel_attention ChannelAttention
3 spatial_attention SpatialAttention
4 global_average_pooling2d GlobalAveragePooling2D
5 dense_2 Dense


In [ ]:
backbone = None

for layer in spatial_model.layers:

    if isinstance(
        layer,
        tf.keras.Model
    ):

        if "mobilenet" in layer.name.lower():

            backbone = layer
            break

if backbone is None:

    raise RuntimeError(
        "MobileNetV2 backbone not found."
    )

print(
    "Backbone recovered:",
    backbone.name
)

Backbone recovered: mobilenetv2_1.00_224


In [ ]:
backbone.trainable = True

for layer in backbone.layers:
    layer.trainable = False

for layer in backbone.layers[-20:]:

    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True

print(
    "Trainable MobileNetV2 layers:",
    sum(
        layer.trainable
        for layer in backbone.layers
    )
)

Trainable MobileNetV2 layers: 13


In [ ]:
PHASE2_LR = 5e-6

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=PHASE2_LR
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
STEPS_PER_EPOCH = math.ceil(
    len(train_window_df) / BATCH_SIZE
)

VALIDATION_STEPS = math.ceil(
    len(val_window_df) / BATCH_SIZE
)

train_ds_phase2 = train_ds.repeat()
val_ds_phase2 = val_ds.repeat()

print("Steps/epoch     :", STEPS_PER_EPOCH)
print("Validation steps:", VALIDATION_STEPS)

Steps/epoch     : 446
Validation steps: 848


In [ ]:
# ============================================================
# PHASE 2 CALLBACKS
# ============================================================

FINAL_BEST = (
    f"{MODEL_DIR}/condition_c_v3_best.keras"
)

callbacks_phase2 = [

    tf.keras.callbacks.ModelCheckpoint(
        FINAL_BEST,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=5e-7,
        verbose=1
    )
]

print("PHASE 2 CALLBACKS READY")
print("Saving best model to:")
print(FINAL_BEST)

PHASE 2 CALLBACKS READY
Saving best model to:
/content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_best.keras


In [ ]:
# ============================================================
# PHASE 2 TRAINING
# ============================================================

history2 = model.fit(

    train_ds_phase2,

    validation_data=val_ds_phase2,

    steps_per_epoch=STEPS_PER_EPOCH,

    validation_steps=VALIDATION_STEPS,

    epochs=8,

    class_weight=class_weights,

    callbacks=callbacks_phase2
)

Epoch 1/8
446/446 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step - accuracy: 0.9514 - loss: 0.1047
Epoch 1: val_loss improved from None to 0.23097, saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_best.keras
446/446 ━━━━━━━━━━━━━━━━━━━━ 451s 626ms/step - accuracy: 0.9484 - loss: 0.1123 - val_accuracy: 0.9171 - val_loss: 0.2310 - learning_rate: 5.0000e-06
Epoch 2/8
446/446 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step - accuracy: 0.9609 - loss: 0.0864
Epoch 2: val_loss did not improve from 0.23097
446/446 ━━━━━━━━━━━━━━━━━━━━ 209s 470ms/step - accuracy: 0.9663 - loss: 0.0814 - val_accuracy: 0.9077 - val_loss: 0.2531 - learning_rate: 5.0000e-06
Epoch 3/8
446/446 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step - accuracy: 0.9657 - loss: 0.0901
Epoch 3: val_loss improved from 0.23097 to 0.22351, saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_best.k

In [ ]:
# ============================================================
# PHASE 2 — VALIDATION EVALUATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# Ground truth
# ------------------------------------------------------------

y_val_true = (
    val_window_df["label"]
    .astype(int)
    .values
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

val_probs_phase2 = model.predict(
    val_ds,
    verbose=1
)

val_fake_probs_phase2 = (
    val_probs_phase2[:, 1]
)

# Temporary 0.50 threshold
val_pred_phase2 = (
    val_fake_probs_phase2 >= 0.50
).astype(int)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

p2_accuracy = accuracy_score(
    y_val_true,
    val_pred_phase2
)

p2_precision = precision_score(
    y_val_true,
    val_pred_phase2,
    zero_division=0
)

p2_recall = recall_score(
    y_val_true,
    val_pred_phase2,
    zero_division=0
)

p2_f1 = f1_score(
    y_val_true,
    val_pred_phase2,
    zero_division=0
)

p2_auc = roc_auc_score(
    y_val_true,
    val_fake_probs_phase2
)

p2_cm = confusion_matrix(
    y_val_true,
    val_pred_phase2
)

print("=" * 70)
print("CONDITION C v3 — PHASE 2 VALIDATION")
print("=" * 70)

print(f"Accuracy  : {p2_accuracy:.4f}")
print(f"Precision : {p2_precision:.4f}")
print(f"Recall    : {p2_recall:.4f}")
print(f"F1-score  : {p2_f1:.4f}")
print(f"ROC-AUC   : {p2_auc:.4f}")

print("\nConfusion Matrix:")
print(p2_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_val_true,
        val_pred_phase2,
        target_names=[
            "REAL",
            "FAKE"
        ],
        digits=4,
        zero_division=0
    )
)

NameError: name 'val_window_df' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import math
import numpy as np
import pandas as pd
import tensorflow as tf

BASE = "/content/drive/MyDrive/deepfake_v2"

PROCESSED_ROOT = f"{BASE}/final_mtcnn_windows"
MODEL_DIR = f"{BASE}/deepfake_models"

TRAIN_DIR = f"{PROCESSED_ROOT}/train"
VAL_DIR   = f"{PROCESSED_ROOT}/val"
TEST_DIR  = f"{PROCESSED_ROOT}/test"

FINAL_BEST = (
    f"{MODEL_DIR}/condition_c_v3_best.keras"
)

print("Final Phase 2 model exists:", os.path.exists(FINAL_BEST))

Mounted at /content/drive
Final Phase 2 model exists: True


In [ ]:
train_window_df = pd.read_csv(
    f"{PROCESSED_ROOT}/train_window_metadata.csv"
)

val_window_df = pd.read_csv(
    f"{PROCESSED_ROOT}/val_window_metadata.csv"
)

test_window_df = pd.read_csv(
    f"{PROCESSED_ROOT}/test_window_metadata.csv"
)

print("Train windows:", len(train_window_df))
print("Val windows  :", len(val_window_df))
print("Test windows :", len(test_window_df))

Train windows: 1783
Val windows  : 3391
Test windows : 3415


In [ ]:
from keras.saving import register_keras_serializable


@register_keras_serializable()
class ChannelAttention(tf.keras.layers.Layer):

    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):

        channels = int(input_shape[-1])
        hidden = max(channels // self.ratio, 1)

        self.fc1 = tf.keras.layers.Dense(
            hidden,
            activation="relu"
        )

        self.fc2 = tf.keras.layers.Dense(
            channels
        )

        super().build(input_shape)

    def call(self, x):

        avg = tf.reduce_mean(
            x,
            axis=[1, 2],
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=[1, 2],
            keepdims=True
        )

        attention = tf.nn.sigmoid(
            self.fc2(self.fc1(avg))
            +
            self.fc2(self.fc1(mx))
        )

        return x * attention

    def get_config(self):

        config = super().get_config()

        config.update({
            "ratio": self.ratio
        })

        return config


@register_keras_serializable()
class SpatialAttention(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):

        self.conv = tf.keras.layers.Conv2D(
            1,
            kernel_size=7,
            padding="same",
            activation="sigmoid"
        )

        super().build(input_shape)

    def call(self, x):

        avg = tf.reduce_mean(
            x,
            axis=-1,
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=-1,
            keepdims=True
        )

        concat = tf.concat(
            [avg, mx],
            axis=-1
        )

        return x * self.conv(concat)

In [ ]:
model = tf.keras.models.load_model(
    FINAL_BEST,
    custom_objects={
        "ChannelAttention": ChannelAttention,
        "SpatialAttention": SpatialAttention
    },
    compile=False,
    safe_mode=False
)

print("MODEL RELOADED")

print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

MODEL RELOADED
Input shape : (None, 15, 224, 224, 3)
Output shape: (None, 2)


In [ ]:
val_files = sorted(
    glob.glob(
        f"{VAL_DIR}/*.npz"
    )
)

print("Validation files:", len(val_files))

Validation files: 50


In [ ]:
def npz_window_generator(file_list):

    for file_path in file_list:

        data = np.load(file_path)

        X = data["X"]
        y = data["y"]

        for i in range(len(y)):

            yield (
                X[i].astype(np.float32) / 255.0,
                np.int32(y[i])
            )

In [ ]:
OUTPUT_SIGNATURE = (
    tf.TensorSpec(
        shape=(15, 224, 224, 3),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(),
        dtype=tf.int32
    )
)

BATCH_SIZE = 4

val_ds = tf.data.Dataset.from_generator(
    lambda: npz_window_generator(val_files),
    output_signature=OUTPUT_SIGNATURE
)

val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("VAL DATASET READY")

VAL DATASET READY


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_val_true = (
    val_window_df["label"]
    .astype(int)
    .values
)

val_probs_phase2 = model.predict(
    val_ds,
    verbose=1
)

val_fake_probs_phase2 = (
    val_probs_phase2[:, 1]
)

val_pred_phase2 = (
    val_fake_probs_phase2 >= 0.50
).astype(int)

p2_accuracy = accuracy_score(
    y_val_true,
    val_pred_phase2
)

p2_precision = precision_score(
    y_val_true,
    val_pred_phase2,
    zero_division=0
)

p2_recall = recall_score(
    y_val_true,
    val_pred_phase2,
    zero_division=0
)

p2_f1 = f1_score(
    y_val_true,
    val_pred_phase2,
    zero_division=0
)

p2_auc = roc_auc_score(
    y_val_true,
    val_fake_probs_phase2
)

p2_cm = confusion_matrix(
    y_val_true,
    val_pred_phase2
)

print("=" * 70)
print("CONDITION C v3 — PHASE 2 VALIDATION")
print("=" * 70)

print(f"Accuracy  : {p2_accuracy:.4f}")
print(f"Precision : {p2_precision:.4f}")
print(f"Recall    : {p2_recall:.4f}")
print(f"F1-score  : {p2_f1:.4f}")
print(f"ROC-AUC   : {p2_auc:.4f}")

print("\nConfusion Matrix:")
print(p2_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_val_true,
        val_pred_phase2,
        target_names=[
            "REAL",
            "FAKE"
        ],
        digits=4,
        zero_division=0
    )
)

848/848 ━━━━━━━━━━━━━━━━━━━━ 235s 255ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


CONDITION C v3 — PHASE 2 VALIDATION
Accuracy  : 0.9304
Precision : 0.9043
Recall    : 0.9043
F1-score  : 0.9043
ROC-AUC   : 0.9764

Confusion Matrix:
[[2040  118]
 [ 118 1115]]

Classification Report:
              precision    recall  f1-score   support

        REAL     0.9453    0.9453    0.9453      2158
        FAKE     0.9043    0.9043    0.9043      1233

    accuracy                         0.9304      3391
   macro avg     0.9248    0.9248    0.9248      3391
weighted avg     0.9304    0.9304    0.9304      3391



In [ ]:
# ============================================================
# THRESHOLD SWEEP — VALIDATION SET
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

thresholds = np.arange(
    0.30,
    0.76,
    0.05
)

threshold_results = []

for threshold in thresholds:

    pred = (
        val_fake_probs_phase2 >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_val_true,
        pred
    )

    precision = precision_score(
        y_val_true,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val_true,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val_true,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(
        y_val_true,
        pred
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "accuracy": accuracy,
        "precision_fake": precision,
        "recall_fake": recall,
        "f1_fake": f1,
        "specificity_real": specificity,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    })


threshold_df = pd.DataFrame(
    threshold_results
)

print("=" * 90)
print("VALIDATION THRESHOLD SWEEP")
print("=" * 90)

print(
    threshold_df[
        [
            "threshold",
            "accuracy",
            "precision_fake",
            "recall_fake",
            "f1_fake",
            "specificity_real"
        ]
    ].round(4).to_string(
        index=False
    )
)

VALIDATION THRESHOLD SWEEP
 threshold  accuracy  precision_fake  recall_fake  f1_fake  specificity_real
      0.30    0.9204          0.8650       0.9254   0.8942            0.9175
      0.35    0.9227          0.8749       0.9189   0.8964            0.9249
      0.40    0.9254          0.8852       0.9132   0.8990            0.9323
      0.45    0.9289          0.8962       0.9100   0.9030            0.9398
      0.50    0.9304          0.9043       0.9043   0.9043            0.9453
      0.55    0.9310          0.9084       0.9011   0.9047            0.9481
      0.60    0.9322          0.9148       0.8970   0.9058            0.9523
      0.65    0.9334          0.9220       0.8921   0.9068            0.9569
      0.70    0.9336          0.9278       0.8865   0.9067            0.9606
      0.75    0.9336          0.9330       0.8808   0.9061            0.9639


In [ ]:
# ============================================================
# FREEZE FINAL WINDOW-LEVEL THRESHOLD
# ============================================================

FINAL_THRESHOLD = 0.65

print("=" * 70)
print("FINAL WINDOW-LEVEL DECISION THRESHOLD")
print("=" * 70)
print(f"Threshold: {FINAL_THRESHOLD:.2f}")
print("Selection basis: highest validation FAKE F1")

FINAL WINDOW-LEVEL DECISION THRESHOLD
Threshold: 0.65
Selection basis: highest validation FAKE F1


In [ ]:
import json
import os

RESULT_DIR = f"{BASE}/final_training_results"
os.makedirs(RESULT_DIR, exist_ok=True)

threshold_config = {
    "model": "condition_c_v3_best.keras",
    "window_threshold": FINAL_THRESHOLD,
    "selection_set": "validation",
    "selection_metric": "fake_f1",
    "validation_accuracy": 0.9334,
    "validation_precision_fake": 0.9220,
    "validation_recall_fake": 0.8921,
    "validation_f1_fake": 0.9068,
    "validation_specificity_real": 0.9569
}

with open(
    f"{RESULT_DIR}/final_threshold.json",
    "w"
) as f:
    json.dump(
        threshold_config,
        f,
        indent=4
    )

print("Threshold configuration saved.")

Threshold configuration saved.


In [ ]:
# ============================================================
# FINAL TEST — CONDITION C v3
# ============================================================

import glob
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

FINAL_THRESHOLD = 0.65

test_files = sorted(
    glob.glob(
        f"{TEST_DIR}/*.npz"
    )
)

test_ds = tf.data.Dataset.from_generator(
    lambda: npz_window_generator(test_files),
    output_signature=OUTPUT_SIGNATURE
)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

y_test_true = (
    test_window_df["label"]
    .astype(int)
    .values
)

test_probs = model.predict(
    test_ds,
    verbose=1
)

test_fake_probs = test_probs[:, 1]

y_test_pred = (
    test_fake_probs >= FINAL_THRESHOLD
).astype(int)

assert len(y_test_pred) == len(y_test_true)

accuracy = accuracy_score(
    y_test_true,
    y_test_pred
)

precision = precision_score(
    y_test_true,
    y_test_pred,
    zero_division=0
)

recall = recall_score(
    y_test_true,
    y_test_pred,
    zero_division=0
)

f1 = f1_score(
    y_test_true,
    y_test_pred,
    zero_division=0
)

auc = roc_auc_score(
    y_test_true,
    test_fake_probs
)

cm = confusion_matrix(
    y_test_true,
    y_test_pred
)

tn, fp, fn, tp = cm.ravel()

specificity = (
    tn / (tn + fp)
)

print("=" * 70)
print("CONDITION C v3 — FINAL TEST")
print("=" * 70)

print(f"Threshold   : {FINAL_THRESHOLD:.2f}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"F1-score    : {f1:.4f}")
print(f"ROC-AUC     : {auc:.4f}")
print(f"Specificity : {specificity:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test_true,
        y_test_pred,
        target_names=[
            "REAL",
            "FAKE"
        ],
        digits=4,
        zero_division=0
    )
)

854/854 ━━━━━━━━━━━━━━━━━━━━ 187s 213ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


CONDITION C v3 — FINAL TEST
Threshold   : 0.65
Accuracy    : 0.8548
Precision   : 0.7829
Recall      : 0.8563
F1-score    : 0.8179
ROC-AUC     : 0.9253
Specificity : 0.8538

Confusion Matrix:
[[1805  309]
 [ 187 1114]]

Classification Report:
              precision    recall  f1-score   support

        REAL     0.9061    0.8538    0.8792      2114
        FAKE     0.7829    0.8563    0.8179      1301

    accuracy                         0.8548      3415
   macro avg     0.8445    0.8550    0.8486      3415
weighted avg     0.8592    0.8548    0.8559      3415



In [ ]:
RESULT_DIR = (
    f"{BASE}/final_training_results"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

test_predictions_df = (
    test_window_df.copy()
)

test_predictions_df[
    "fake_probability"
] = test_fake_probs

test_predictions_df[
    "prediction"
] = y_test_pred

test_predictions_df.to_csv(
    f"{RESULT_DIR}/test_window_predictions_v3.csv",
    index=False
)

print("TEST predictions saved.")

TEST predictions saved.


In [ ]:
test_result_summary = pd.DataFrame([
    {
        "model": "Condition C v3",
        "threshold": 0.65,
        "accuracy": 0.8548,
        "precision_fake": 0.7829,
        "recall_fake": 0.8563,
        "f1_fake": 0.8179,
        "roc_auc": 0.9253,
        "specificity_real": 0.8538,
        "tn": 1805,
        "fp": 309,
        "fn": 187,
        "tp": 1114
    }
])

test_result_summary.to_csv(
    f"{RESULT_DIR}/condition_c_v3_final_test_results.csv",
    index=False
)

print("Final TEST summary saved.")

Final TEST summary saved.


In [ ]:
RESULT_DIR = f"{BASE}/final_training_results"

val_predictions_df = val_window_df.copy()

val_predictions_df["fake_probability"] = (
    val_fake_probs_phase2
)

val_predictions_df["prediction_065"] = (
    val_fake_probs_phase2 >= 0.65
).astype(int)

val_predictions_df.to_csv(
    f"{RESULT_DIR}/val_window_predictions_v3.csv",
    index=False
)

print("Validation predictions saved.")

Validation predictions saved.


In [ ]:
# ============================================================
# VIDEO-LEVEL AGGREGATION TUNING — VALIDATION ONLY
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

WINDOW_THRESHOLD = 0.65
WINDOW_STRIDE_SECONDS = 1.0


def true_video_class(fake_type):

    fake_type = str(fake_type).lower()

    if fake_type == "real":
        return "REAL"

    if fake_type == "partial":
        return "PARTIAL"

    if fake_type in {
        "face_swap",
        "ai_generated"
    }:
        return "FAKE"

    raise ValueError(
        f"Unknown fake type: {fake_type}"
    )


def sustained_fake_mask(
    group,
    min_run
):

    group = (
        group
        .sort_values("window_start")
        .reset_index(drop=True)
    )

    starts = (
        group["window_start"]
        .to_numpy()
    )

    raw_fake = (
        group["fake_probability"]
        .to_numpy()
        >=
        WINDOW_THRESHOLD
    )

    keep = np.zeros(
        len(group),
        dtype=bool
    )

    run = []

    for i in range(len(group)):

        if raw_fake[i]:

            if not run:

                run = [i]

            else:

                previous = run[-1]

                if (
                    starts[i]
                    -
                    starts[previous]
                    <=
                    WINDOW_STRIDE_SECONDS
                    + 0.1
                ):

                    run.append(i)

                else:

                    if len(run) >= min_run:
                        keep[run] = True

                    run = [i]

        else:

            if len(run) >= min_run:
                keep[run] = True

            run = []

    if len(run) >= min_run:
        keep[run] = True

    return keep


def predict_video_class(
    group,
    min_run,
    full_fake_threshold
):

    mask = sustained_fake_mask(
        group,
        min_run
    )

    total = len(mask)

    fake_windows = int(
        mask.sum()
    )

    coverage = (
        fake_windows / total
        if total > 0
        else 0.0
    )

    if fake_windows == 0:

        verdict = "REAL"

    elif coverage >= full_fake_threshold:

        verdict = "FAKE"

    else:

        verdict = "PARTIAL"

    return verdict, coverage

In [ ]:
results = []

for min_run in [
    1,
    2,
    3,
    4
]:

    for full_fake_threshold in np.arange(
        0.30,
        0.91,
        0.05
    ):

        y_true = []
        y_pred = []

        for video_id, group in (
            val_predictions_df
            .groupby("video_id")
        ):

            truth = true_video_class(
                group["fake_type"].iloc[0]
            )

            pred, coverage = (
                predict_video_class(
                    group,
                    min_run,
                    float(
                        full_fake_threshold
                    )
                )
            )

            y_true.append(
                truth
            )

            y_pred.append(
                pred
            )

        accuracy = accuracy_score(
            y_true,
            y_pred
        )

        macro_f1 = f1_score(
            y_true,
            y_pred,
            average="macro"
        )

        results.append({
            "min_run":
                min_run,

            "full_fake_threshold":
                round(
                    float(
                        full_fake_threshold
                    ),
                    2
                ),

            "accuracy":
                accuracy,

            "macro_f1":
                macro_f1
        })


aggregation_df = pd.DataFrame(
    results
)

aggregation_df = (
    aggregation_df
    .sort_values(
        [
            "macro_f1",
            "accuracy"
        ],
        ascending=False
    )
)

print(
    aggregation_df.head(20)
)

    min_run  full_fake_threshold  accuracy  macro_f1
39        4                 0.30      0.88  0.859436
40        4                 0.35      0.88  0.859436
41        4                 0.40      0.88  0.859436
26        3                 0.30      0.88  0.851648
42        4                 0.45      0.86  0.842660
43        4                 0.50      0.86  0.842660
44        4                 0.55      0.86  0.842660
45        4                 0.60      0.86  0.842660
46        4                 0.65      0.86  0.842660
27        3                 0.35      0.86  0.835210
28        3                 0.40      0.86  0.835210
29        3                 0.45      0.86  0.835210
30        3                 0.50      0.86  0.835210
31        3                 0.55      0.86  0.835210
32        3                 0.60      0.86  0.835210
33        3                 0.65      0.86  0.835210
47        4                 0.70      0.84  0.826127
34        3                 0.70      0.84  0.

In [ ]:
best = aggregation_df.iloc[0]

BEST_MIN_RUN = int(
    best["min_run"]
)

FULL_FAKE_COVERAGE = float(
    best["full_fake_threshold"]
)

print("=" * 70)
print("FINAL VIDEO AGGREGATION CONFIG")
print("=" * 70)

print(
    "Minimum consecutive fake windows:",
    BEST_MIN_RUN
)

print(
    "Full fake coverage threshold:",
    FULL_FAKE_COVERAGE
)

print(
    "Validation macro F1:",
    round(
        float(
            best["macro_f1"]
        ),
        4
    )
)

print(
    "Validation accuracy:",
    round(
        float(
            best["accuracy"]
        ),
        4
    )
)

FINAL VIDEO AGGREGATION CONFIG
Minimum consecutive fake windows: 4
Full fake coverage threshold: 0.3
Validation macro F1: 0.8594
Validation accuracy: 0.88


In [ ]:
import json
import os

deployment_config = {

    "window_seconds":
        2.0,

    "window_stride_seconds":
        1.0,

    "frames_per_window":
        15,

    "image_size":
        224,

    "minimum_detected_faces":
        8,

    "mtcnn_confidence":
        0.80,

    "window_fake_threshold":
        0.65,

    "minimum_consecutive_fake_windows":
        BEST_MIN_RUN,

    "full_fake_coverage_threshold":
        FULL_FAKE_COVERAGE
}

CONFIG_PATH = (
    f"{RESULT_DIR}/"
    "condition_c_v3_deployment_config.json"
)

with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        deployment_config,
        f,
        indent=4
    )

print(
    "Saved:",
    CONFIG_PATH
)

print(
    json.dumps(
        deployment_config,
        indent=4
    )
)

Saved: /content/drive/MyDrive/deepfake_v2/final_training_results/condition_c_v3_deployment_config.json
{
    "window_seconds": 2.0,
    "window_stride_seconds": 1.0,
    "frames_per_window": 15,
    "image_size": 224,
    "minimum_detected_faces": 8,
    "mtcnn_confidence": 0.8,
    "window_fake_threshold": 0.65,
    "minimum_consecutive_fake_windows": 4,
    "full_fake_coverage_threshold": 0.3
}


In [ ]:
import hashlib

MODEL = "/content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_best.keras"

with open(MODEL, "rb") as f:
    print(hashlib.sha256(f.read()).hexdigest())

9338979ac81d3114658f08ee0ea5985cbf13e9855f595c79fa08b5558c0c6224


In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


In [ ]:
BASE = (
    "/content/drive/MyDrive/deepfake_v2"
)

HARD_REAL_DIR = (
    f"{BASE}/hard_negative_real"
)

HARD_MANIFEST = (
    f"{BASE}/hard_negative_manifest_v3.csv"
)

HARD_OUTPUT = (
    f"{BASE}/hard_negative_windows_v3"
)

In [ ]:
import os
import pandas as pd

print(
    "Video folder:",
    os.path.exists(
        HARD_REAL_DIR
    )
)

print(
    "Manifest:",
    os.path.exists(
        HARD_MANIFEST
    )
)

hard_df = pd.read_csv(
    HARD_MANIFEST
)

display(
    hard_df[
        [
            "video_id",
            "hard_negative_split",
            "duration_sec"
        ]
    ]
)

print()
print(
    hard_df[
        "hard_negative_split"
    ].value_counts()
)

Video folder: True
Manifest: True


,video_id,hard_negative_split,duration_sec
0,hardreal_001,train,11.067
1,hardreal_002,train,11.970
2,hardreal_003,train,24.792
3,hardreal_004,train,55.433
4,hardreal_005,holdout,31.767
5,hardreal_006,holdout,34.033
6,hardreal_007,train,60.067
7,hardreal_008,train,40.267
8,hardreal_009,train,59.059
9,hardreal_010,holdout,31.900



hard_negative_split
train      16
holdout     5
Name: count, dtype: int64


In [ ]:
!pip install -q mtcnn opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.3 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from mtcnn import MTCNN

In [ ]:
BASE = "/content/drive/MyDrive/deepfake_v2"

HARD_REAL_DIR = f"{BASE}/hard_negative_real"
HARD_MANIFEST = f"{BASE}/hard_negative_manifest_v3.csv"

HARD_OUTPUT = f"{BASE}/hard_negative_windows_v3"

TRAIN_OUTPUT = f"{HARD_OUTPUT}/train"
HOLDOUT_OUTPUT = f"{HARD_OUTPUT}/holdout"

os.makedirs(TRAIN_OUTPUT, exist_ok=True)
os.makedirs(HOLDOUT_OUTPUT, exist_ok=True)

# ============================================================
# EXACT CONDITION C v3 PREPROCESSING SETTINGS
# ============================================================

WINDOW_SECONDS = 2.0
STRIDE_SECONDS = 1.0
FRAMES_PER_WINDOW = 15

IMAGE_SIZE = 224
MINIMUM_FACES = 8
MTCNN_CONFIDENCE = 0.80

print("=" * 70)
print("HARD-NEGATIVE PREPROCESSING CONFIG")
print("=" * 70)
print("Video dir        :", HARD_REAL_DIR)
print("Manifest         :", HARD_MANIFEST)
print("Output           :", HARD_OUTPUT)
print("Window seconds   :", WINDOW_SECONDS)
print("Stride seconds   :", STRIDE_SECONDS)
print("Frames/window    :", FRAMES_PER_WINDOW)
print("Image size       :", IMAGE_SIZE)
print("Minimum faces    :", MINIMUM_FACES)
print("MTCNN confidence :", MTCNN_CONFIDENCE)

HARD-NEGATIVE PREPROCESSING CONFIG
Video dir        : /content/drive/MyDrive/deepfake_v2/hard_negative_real
Manifest         : /content/drive/MyDrive/deepfake_v2/hard_negative_manifest_v3.csv
Output           : /content/drive/MyDrive/deepfake_v2/hard_negative_windows_v3
Window seconds   : 2.0
Stride seconds   : 1.0
Frames/window    : 15
Image size       : 224
Minimum faces    : 8
MTCNN confidence : 0.8


In [ ]:
hard_df = pd.read_csv(HARD_MANIFEST)

hard_df["colab_path"] = hard_df["filename"].apply(
    lambda filename: os.path.join(
        HARD_REAL_DIR,
        filename
    )
)

print(hard_df["hard_negative_split"].value_counts())

display(
    hard_df[
        [
            "video_id",
            "filename",
            "hard_negative_split",
            "duration_sec",
            "colab_path"
        ]
    ]
)

hard_negative_split
train      16
holdout     5
Name: count, dtype: int64


,video_id,filename,hard_negative_split,duration_sec,colab_path
0,hardreal_001,hardreal_001.mp4,train,11.067,/content/drive/MyDrive/deepfake_v2/hard_negati...
1,hardreal_002,hardreal_002.mp4,train,11.970,/content/drive/MyDrive/deepfake_v2/hard_negati...
2,hardreal_003,hardreal_003.mp4,train,24.792,/content/drive/MyDrive/deepfake_v2/hard_negati...
3,hardreal_004,hardreal_004.mp4,train,55.433,/content/drive/MyDrive/deepfake_v2/hard_negati...
4,hardreal_005,hardreal_005.mp4,holdout,31.767,/content/drive/MyDrive/deepfake_v2/hard_negati...
5,hardreal_006,hardreal_006.mp4,holdout,34.033,/content/drive/MyDrive/deepfake_v2/hard_negati...
6,hardreal_007,hardreal_007.mp4,train,60.067,/content/drive/MyDrive/deepfake_v2/hard_negati...
7,hardreal_008,hardreal_008.mp4,train,40.267,/content/drive/MyDrive/deepfake_v2/hard_negati...
8,hardreal_009,hardreal_009.mp4,train,59.059,/content/drive/MyDrive/deepfake_v2/hard_negati...
9,hardreal_010,hardreal_010.mp4,holdout,31.900,/content/drive/MyDrive/deepfake_v2/hard_negati...


In [ ]:
missing = hard_df[
    ~hard_df["colab_path"].apply(os.path.exists)
]

print("Missing videos:", len(missing))

if len(missing):
    display(missing)
else:
    print("All 21 videos found successfully.")

Missing videos: 0
All 21 videos found successfully.


In [ ]:
detector = MTCNN()

print("MTCNN initialized successfully.")

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.


MTCNN initialized successfully.


In [ ]:
def safe_mtcnn_detect(detector, rgb):

    try:
        detections = detector.detect_faces(rgb)

        if detections is None:
            return []

        return detections

    except ValueError as e:

        text = str(e).lower()

        if (
            "empty output" in text
            or "shape=(0" in text
            or "incompatible shapes" in text
        ):
            return []

        raise

    except Exception as e:

        print("MTCNN warning:", str(e))
        return []

In [ ]:
def extract_largest_face(
    rgb,
    detector,
    image_size=224,
    confidence_threshold=0.80
):

    detections = safe_mtcnn_detect(
        detector,
        rgb
    )

    if not detections:
        return None

    valid = []

    for detection in detections:

        box = detection.get("box")

        confidence = float(
            detection.get(
                "confidence",
                0.0
            )
        )

        if not box:
            continue

        x, y, w, h = box

        if w <= 0 or h <= 0:
            continue

        if confidence < confidence_threshold:
            continue

        valid.append(detection)

    if not valid:
        return None

    best = max(
        valid,
        key=lambda detection:
            detection["box"][2]
            *
            detection["box"][3]
    )

    x, y, w, h = best["box"]

    frame_height, frame_width = rgb.shape[:2]

    x = max(0, int(x))
    y = max(0, int(y))

    x2 = min(
        frame_width,
        x + int(w)
    )

    y2 = min(
        frame_height,
        y + int(h)
    )

    if x2 <= x or y2 <= y:
        return None

    face = rgb[
        y:y2,
        x:x2
    ]

    if face.size == 0:
        return None

    face = cv2.resize(
        face,
        (image_size, image_size),
        interpolation=cv2.INTER_AREA
    )

    return face.astype(np.uint8)

In [ ]:
def extract_window(
    video_path,
    start_sec,
    end_sec,
    detector
):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None, 0

    timestamps = np.linspace(
        start_sec,
        end_sec,
        FRAMES_PER_WINDOW,
        endpoint=False
    )

    detected_faces = {}

    for position, timestamp in enumerate(timestamps):

        cap.set(
            cv2.CAP_PROP_POS_MSEC,
            float(timestamp * 1000.0)
        )

        ok, frame = cap.read()

        if not ok:
            continue

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        face = extract_largest_face(
            rgb,
            detector,
            IMAGE_SIZE,
            MTCNN_CONFIDENCE
        )

        if face is not None:
            detected_faces[position] = face

    cap.release()

    detected_count = len(detected_faces)

    if detected_count < MINIMUM_FACES:
        return None, detected_count

    # ========================================================
    # SAME NEAREST-VALID-FACE FILLING USED IN TRAINING
    # ========================================================

    valid_positions = sorted(
        detected_faces.keys()
    )

    sequence = []

    for position in range(FRAMES_PER_WINDOW):

        if position in detected_faces:

            sequence.append(
                detected_faces[position]
            )

        else:

            nearest = min(
                valid_positions,
                key=lambda valid_position:
                    abs(
                        valid_position
                        -
                        position
                    )
            )

            sequence.append(
                detected_faces[nearest]
            )

    sequence = np.stack(
        sequence,
        axis=0
    )

    if sequence.shape != (
        FRAMES_PER_WINDOW,
        IMAGE_SIZE,
        IMAGE_SIZE,
        3
    ):
        return None, detected_count

    return sequence.astype(np.uint8), detected_count

In [ ]:
sample_row = hard_df[
    hard_df["hard_negative_split"] == "train"
].iloc[0]

video_path = sample_row["colab_path"]
video_id = sample_row["video_id"]

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

duration = (
    frame_count / fps
    if fps > 0
    else 0
)

cap.release()

print("Video:", video_id)
print("Duration:", duration)

window_starts = np.arange(
    0.0,
    duration
    -
    WINDOW_SECONDS
    +
    1e-6,
    STRIDE_SECONDS
)

accepted = 0
rejected = 0

for start in window_starts[:10]:

    end = start + WINDOW_SECONDS

    sequence, face_count = extract_window(
        video_path,
        start,
        end,
        detector
    )

    if sequence is None:

        rejected += 1

        print(
            f"{start:.1f}-{end:.1f}s | "
            f"faces={face_count}/15 | REJECTED"
        )

    else:

        accepted += 1

        print(
            f"{start:.1f}-{end:.1f}s | "
            f"faces={face_count}/15 | ACCEPTED | "
            f"shape={sequence.shape}"
        )

print()
print("Accepted:", accepted)
print("Rejected:", rejected)

Video: hardreal_001
Duration: 11.066666666666666
0.0-2.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
1.0-3.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
2.0-4.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
3.0-5.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
4.0-6.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
5.0-7.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
6.0-8.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
7.0-9.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
8.0-10.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)
9.0-11.0s | faces=15/15 | ACCEPTED | shape=(15, 224, 224, 3)

Accepted: 10
Rejected: 0


In [ ]:
def process_hard_negative_split(
    dataframe,
    split_name
):

    split_df = dataframe[
        dataframe[
            "hard_negative_split"
        ]
        ==
        split_name
    ].copy()

    output_dir = (
        TRAIN_OUTPUT
        if split_name == "train"
        else HOLDOUT_OUTPUT
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    metadata = []

    total_videos = len(split_df)

    total_candidate_windows = 0
    total_saved_windows = 0
    total_rejected_windows = 0

    print()
    print("=" * 80)
    print(
        f"PROCESSING HARD-NEGATIVE {split_name.upper()} SPLIT"
    )
    print("=" * 80)

    for video_index, (_, row) in enumerate(
        split_df.iterrows(),
        start=1
    ):

        video_id = str(row["video_id"])
        video_path = row["colab_path"]

        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():

            print(
                f"[{video_index}/{total_videos}] "
                f"{video_id} | COULD NOT OPEN"
            )

            continue

        fps = float(
            cap.get(
                cv2.CAP_PROP_FPS
            )
        )

        frame_count = int(
            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
        )

        duration = (
            frame_count / fps
            if fps > 0
            else 0.0
        )

        cap.release()

        if duration < WINDOW_SECONDS:

            print(
                f"[{video_index}/{total_videos}] "
                f"{video_id} | TOO SHORT"
            )

            continue

        starts = np.arange(
            0.0,
            duration
            -
            WINDOW_SECONDS
            +
            1e-6,
            STRIDE_SECONDS
        )

        video_saved = 0
        video_rejected = 0

        for window_number, start in enumerate(starts):

            end = (
                float(start)
                +
                WINDOW_SECONDS
            )

            total_candidate_windows += 1

            sequence, face_count = extract_window(
                video_path,
                float(start),
                float(end),
                detector
            )

            if sequence is None:

                total_rejected_windows += 1
                video_rejected += 1

                continue

            window_id = (
                f"{video_id}"
                f"_w{window_number:04d}"
            )

            output_path = os.path.join(
                output_dir,
                window_id + ".npz"
            )

            np.savez_compressed(
                output_path,
                X=sequence,
                y=np.int64(0)
            )

            metadata.append({

                "window_id":
                    window_id,

                "video_id":
                    video_id,

                "label":
                    0,

                "fake_type":
                    "real_hard_negative",

                "hard_negative_split":
                    split_name,

                "start_sec":
                    round(
                        float(start),
                        3
                    ),

                "end_sec":
                    round(
                        float(end),
                        3
                    ),

                "face_count":
                    int(
                        face_count
                    ),

                "window_path":
                    output_path
            })

            total_saved_windows += 1
            video_saved += 1

        print(
            f"[{video_index}/{total_videos}] "
            f"{video_id} | "
            f"duration={duration:.1f}s | "
            f"saved={video_saved} | "
            f"rejected={video_rejected}"
        )

    metadata_df = pd.DataFrame(
        metadata
    )

    metadata_path = os.path.join(
        HARD_OUTPUT,
        f"{split_name}_metadata.csv"
    )

    metadata_df.to_csv(
        metadata_path,
        index=False
    )

    print()
    print("=" * 80)
    print(
        f"{split_name.upper()} HARD-NEGATIVE PREPROCESSING COMPLETE"
    )
    print("=" * 80)

    print(
        "Videos:",
        total_videos
    )

    print(
        "Candidate windows:",
        total_candidate_windows
    )

    print(
        "Saved windows:",
        total_saved_windows
    )

    print(
        "Rejected windows:",
        total_rejected_windows
    )

    print(
        "Metadata:",
        metadata_path
    )

    return metadata_df

In [ ]:
hard_train_window_df = process_hard_negative_split(
    hard_df,
    "train"
)


PROCESSING HARD-NEGATIVE TRAIN SPLIT
[1/16] hardreal_001 | duration=11.1s | saved=10 | rejected=0
[2/16] hardreal_002 | duration=12.0s | saved=10 | rejected=0
[3/16] hardreal_003 | duration=24.8s | saved=23 | rejected=0
[4/16] hardreal_004 | duration=55.4s | saved=50 | rejected=4
[5/16] hardreal_007 | duration=60.1s | saved=49 | rejected=10
[6/16] hardreal_008 | duration=40.3s | saved=39 | rejected=0
[7/16] hardreal_009 | duration=59.1s | saved=48 | rejected=10
[8/16] hardreal_011 | duration=31.1s | saved=26 | rejected=4
[9/16] hardreal_012 | duration=7.6s | saved=6 | rejected=0
[10/16] hardreal_013 | duration=56.0s | saved=55 | rejected=0
[11/16] hardreal_014 | duration=20.3s | saved=18 | rejected=1
[12/16] hardreal_016 | duration=33.0s | saved=28 | rejected=3
[13/16] hardreal_017 | duration=36.1s | saved=31 | rejected=4
[14/16] hardreal_018 | duration=33.8s | saved=25 | rejected=7
[15/16] hardreal_019 | duration=121.7s | saved=116 | rejected=4
[16/16] hardreal_021 | duration=11.8s |

In [ ]:
print(
    "Hard-negative TRAIN windows:",
    len(
        hard_train_window_df
    )
)

print(
    hard_train_window_df[
        "video_id"
    ].nunique()
)

display(
    hard_train_window_df.head()
)

Hard-negative TRAIN windows: 544
16


,window_id,video_id,label,fake_type,hard_negative_split,start_sec,end_sec,face_count,window_path
0,hardreal_001_w0000,hardreal_001,0,real_hard_negative,train,0.0,2.0,15,/content/drive/MyDrive/deepfake_v2/hard_negati...
1,hardreal_001_w0001,hardreal_001,0,real_hard_negative,train,1.0,3.0,15,/content/drive/MyDrive/deepfake_v2/hard_negati...
2,hardreal_001_w0002,hardreal_001,0,real_hard_negative,train,2.0,4.0,15,/content/drive/MyDrive/deepfake_v2/hard_negati...
3,hardreal_001_w0003,hardreal_001,0,real_hard_negative,train,3.0,5.0,15,/content/drive/MyDrive/deepfake_v2/hard_negati...
4,hardreal_001_w0004,hardreal_001,0,real_hard_negative,train,4.0,6.0,15,/content/drive/MyDrive/deepfake_v2/hard_negati...


In [ ]:
hard_holdout_window_df = process_hard_negative_split(
    hard_df,
    "holdout"
)


PROCESSING HARD-NEGATIVE HOLDOUT SPLIT
[1/5] hardreal_005 | duration=31.8s | saved=30 | rejected=0
[2/5] hardreal_006 | duration=34.0s | saved=29 | rejected=4
[3/5] hardreal_010 | duration=31.9s | saved=26 | rejected=4
[4/5] hardreal_015 | duration=25.8s | saved=23 | rejected=1
[5/5] hardreal_020 | duration=23.9s | saved=22 | rejected=0

HOLDOUT HARD-NEGATIVE PREPROCESSING COMPLETE
Videos: 5
Candidate windows: 139
Saved windows: 130
Rejected windows: 9
Metadata: /content/drive/MyDrive/deepfake_v2/hard_negative_windows_v3/holdout_metadata.csv


In [ ]:
print("=" * 70)
print("HARD-NEGATIVE WINDOW QA")
print("=" * 70)

print()
print("TRAIN")
print(
    "Videos:",
    hard_train_window_df[
        "video_id"
    ].nunique()
)

print(
    "Windows:",
    len(
        hard_train_window_df
    )
)

print(
    hard_train_window_df[
        "face_count"
    ].describe()
)

print()
print("HOLDOUT")
print(
    "Videos:",
    hard_holdout_window_df[
        "video_id"
    ].nunique()
)

print(
    "Windows:",
    len(
        hard_holdout_window_df
    )
)

print(
    hard_holdout_window_df[
        "face_count"
    ].describe()
)

HARD-NEGATIVE WINDOW QA

TRAIN
Videos: 16
Windows: 544
count    544.000000
mean      14.685662
std        1.121733
min        8.000000
25%       15.000000
50%       15.000000
75%       15.000000
max       15.000000
Name: face_count, dtype: float64

HOLDOUT
Videos: 5
Windows: 130
count    130.000000
mean      14.346154
std        1.205360
min       11.000000
25%       14.000000
50%       15.000000
75%       15.000000
max       15.000000
Name: face_count, dtype: float64


In [ ]:
train_video_ids = set(
    hard_train_window_df[
        "video_id"
    ].unique()
)

holdout_video_ids = set(
    hard_holdout_window_df[
        "video_id"
    ].unique()
)

overlap = (
    train_video_ids
    &
    holdout_video_ids
)

print(
    "Train/Holdout video overlap:",
    overlap
)

assert len(overlap) == 0

print(
    "✅ NO VIDEO-LEVEL LEAKAGE"
)

Train/Holdout video overlap: set()
✅ NO VIDEO-LEVEL LEAKAGE


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/deepfake_v2"

ORIGINAL_TRAIN_META = (
    f"{BASE}/final_mtcnn_windows/"
    "train_window_metadata.csv"
)

HARD_TRAIN_META = (
    f"{BASE}/hard_negative_windows_v3/"
    "train_metadata.csv"
)

print(
    "Original train metadata:",
    os.path.exists(
        ORIGINAL_TRAIN_META
    )
)

print(
    "Hard-negative metadata:",
    os.path.exists(
        HARD_TRAIN_META
    )
)

original_train_df = pd.read_csv(
    ORIGINAL_TRAIN_META
)

hard_train_df = pd.read_csv(
    HARD_TRAIN_META
)

print()
print(
    "Original windows:",
    len(original_train_df)
)

print(
    "Hard-negative REAL windows:",
    len(hard_train_df)
)

Original train metadata: True
Hard-negative metadata: True

Original windows: 1783
Hard-negative REAL windows: 544


In [ ]:
print(
    original_train_df[
        "label"
    ].value_counts()
)

print()

print(
    original_train_df[
        "fake_type"
    ].value_counts()
)

label
1    1088
0     695
Name: count, dtype: int64

fake_type
partial         563
face_swap       470
real            428
ai_generated    322
Name: count, dtype: int64


In [ ]:
SEED = 42

original_real_df = (
    original_train_df[
        original_train_df[
            "label"
        ]
        ==
        0
    ]
    .sample(
        n=min(
            544,
            len(
                original_train_df[
                    original_train_df[
                        "label"
                    ]
                    ==
                    0
                ]
            )
        ),
        random_state=SEED
    )
    .copy()
)

original_fake_df = (
    original_train_df[
        original_train_df[
            "label"
        ]
        ==
        1
    ]
    .copy()
)

hard_train_df = (
    hard_train_df
    .copy()
)

hard_train_df[
    "label"
] = 0

In [ ]:
finetune_df = pd.concat(
    [
        original_real_df,
        original_fake_df,
        hard_train_df
    ],
    ignore_index=True
)

finetune_df = (
    finetune_df
    .sample(
        frac=1.0,
        random_state=SEED
    )
    .reset_index(
        drop=True
    )
)

print("=" * 70)
print("FINE-TUNING DATASET")
print("=" * 70)

print(
    "Total windows:",
    len(finetune_df)
)

print()

print(
    finetune_df[
        "label"
    ].value_counts()
)

FINE-TUNING DATASET
Total windows: 2176

label
0    1088
1    1088
Name: count, dtype: int64


In [ ]:
print(
    original_train_df.columns.tolist()
)

print()

print(
    hard_train_df.columns.tolist()
)

['split', 'video_id', 'language', 'fake_type', 'window_start', 'window_end', 'label', 'face_count']

['window_id', 'video_id', 'label', 'fake_type', 'hard_negative_split', 'start_sec', 'end_sec', 'face_count', 'window_path']


In [ ]:
import os
from pathlib import Path

BASE = "/content/drive/MyDrive/deepfake_v2"

ORIGINAL_WINDOW_ROOT = (
    f"{BASE}/final_mtcnn_windows"
)

print("=" * 80)
print("ORIGINAL WINDOW STORAGE INSPECTION")
print("=" * 80)
print("Root exists:", os.path.exists(ORIGINAL_WINDOW_ROOT))
print()

files_found = []

for root, dirs, files in os.walk(ORIGINAL_WINDOW_ROOT):

    for filename in files:

        path = os.path.join(
            root,
            filename
        )

        files_found.append(path)

print("Total files found:", len(files_found))
print()

print("FIRST 50 FILES:")
print("-" * 80)

for path in files_found[:50]:
    print(path)

print()
print("=" * 80)

# Extension summary
from collections import Counter

extensions = Counter(
    Path(path).suffix.lower()
    for path in files_found
)

print("FILE EXTENSIONS:")
print(extensions)

ORIGINAL WINDOW STORAGE INSPECTION
Root exists: True

Total files found: 336

FIRST 50 FILES:
--------------------------------------------------------------------------------
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train_window_metadata.csv
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/val_window_metadata.csv
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/test_window_metadata.csv
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0001.npz
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0002.npz
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0003.npz
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0004.npz
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0005.npz
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0006.npz
/content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0007.npz
/content/drive/MyDrive/deepfake_v2/final_mtcnn_window

In [ ]:
print("TRAIN metadata first rows:")
display(
    original_train_df.head(10)
)

TRAIN metadata first rows:


,split,video_id,language,fake_type,window_start,window_end,label,face_count
0,train,tamil_010,tamil,real,92.0,94.0,0,15
1,train,tamil_010,tamil,real,93.0,95.0,0,15
2,train,tamil_010,tamil,real,32.0,34.0,0,15
3,train,tamil_010,tamil,real,96.0,98.0,0,15
4,train,sinhala_042,sinhala,real,118.0,120.0,0,15
5,train,sinhala_042,sinhala,real,106.0,108.0,0,15
6,train,sinhala_042,sinhala,real,9.0,11.0,0,11
7,train,sinhala_042,sinhala,real,83.0,85.0,0,15
8,train,sinhala_042,sinhala,real,52.0,54.0,0,15
9,train,sinhala_042,sinhala,real,90.0,92.0,0,15


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

BASE = "/content/drive/MyDrive/deepfake_v2"

WINDOW_ROOT = f"{BASE}/final_mtcnn_windows"

TRAIN_DIR = f"{WINDOW_ROOT}/train"
VAL_DIR   = f"{WINDOW_ROOT}/val"
TEST_DIR  = f"{WINDOW_ROOT}/test"

TRAIN_META = f"{WINDOW_ROOT}/train_window_metadata.csv"
VAL_META   = f"{WINDOW_ROOT}/val_window_metadata.csv"
TEST_META  = f"{WINDOW_ROOT}/test_window_metadata.csv"

HARD_TRAIN_META = (
    f"{BASE}/hard_negative_windows_v3/"
    "train_metadata.csv"
)

print("=" * 70)
print("RUNTIME RECOVERY CHECK")
print("=" * 70)

paths = {
    "BASE": BASE,
    "TRAIN_DIR": TRAIN_DIR,
    "VAL_DIR": VAL_DIR,
    "TEST_DIR": TEST_DIR,
    "TRAIN_META": TRAIN_META,
    "VAL_META": VAL_META,
    "TEST_META": TEST_META,
    "HARD_TRAIN_META": HARD_TRAIN_META,
}

for name, path in paths.items():
    print(
        f"{name:18s}:",
        "✅ FOUND" if os.path.exists(path) else "❌ MISSING"
    )

RUNTIME RECOVERY CHECK
BASE              : ✅ FOUND
TRAIN_DIR         : ✅ FOUND
VAL_DIR           : ✅ FOUND
TEST_DIR          : ✅ FOUND
TRAIN_META        : ✅ FOUND
VAL_META          : ✅ FOUND
TEST_META         : ✅ FOUND
HARD_TRAIN_META   : ✅ FOUND


In [ ]:
original_train_df = pd.read_csv(TRAIN_META)
val_df = pd.read_csv(VAL_META)
test_df = pd.read_csv(TEST_META)
hard_train_df = pd.read_csv(HARD_TRAIN_META)

print("=" * 70)
print("METADATA RELOADED")
print("=" * 70)

print("Original train rows :", len(original_train_df))
print("Validation rows     :", len(val_df))
print("Test rows           :", len(test_df))
print("Hard-negative rows  :", len(hard_train_df))

print("\nOriginal columns:")
print(original_train_df.columns.tolist())

print("\nHard-negative columns:")
print(hard_train_df.columns.tolist())

METADATA RELOADED
Original train rows : 1783
Validation rows     : 3391
Test rows           : 3415
Hard-negative rows  : 544

Original columns:
['split', 'video_id', 'language', 'fake_type', 'window_start', 'window_end', 'label', 'face_count']

Hard-negative columns:
['window_id', 'video_id', 'label', 'fake_type', 'hard_negative_split', 'start_sec', 'end_sec', 'face_count', 'window_path']


In [ ]:
print("=" * 80)
print("ORIGINAL NPZ STORAGE CHECK")
print("=" * 80)

for split_name, folder in [
    ("TRAIN", TRAIN_DIR),
    ("VAL", VAL_DIR),
    ("TEST", TEST_DIR),
]:

    if not os.path.exists(folder):
        print(
            f"{split_name}: FOLDER MISSING -> {folder}"
        )
        continue

    files = sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.endswith(".npz")
    ])

    print(
        f"{split_name}: {len(files)} NPZ files"
    )

print("=" * 80)

ORIGINAL NPZ STORAGE CHECK
TRAIN: 232 NPZ files
VAL: 50 NPZ files
TEST: 51 NPZ files


In [ ]:
train_npz_files = sorted([
    os.path.join(TRAIN_DIR, f)
    for f in os.listdir(TRAIN_DIR)
    if f.endswith(".npz")
])

print(
    "Train NPZ files:",
    len(train_npz_files)
)

for path in train_npz_files[:5]:

    print("\n" + "=" * 80)
    print("FILE:", path)

    with np.load(
        path,
        allow_pickle=True
    ) as data:

        print("Keys:", data.files)

        for key in data.files:

            value = data[key]

            print(
                f"{key:20s} "
                f"shape={value.shape} "
                f"dtype={value.dtype}"
            )

Train NPZ files: 232

FILE: /content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0001.npz
Keys: ['X', 'y', 'starts', 'ends', 'face_counts']
X                    shape=(4, 15, 224, 224, 3) dtype=uint8
y                    shape=(4,) dtype=int8
starts               shape=(4,) dtype=float32
ends                 shape=(4,) dtype=float32
face_counts          shape=(4,) dtype=int8

FILE: /content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0002.npz
Keys: ['X', 'y', 'starts', 'ends', 'face_counts']
X                    shape=(10, 15, 224, 224, 3) dtype=uint8
y                    shape=(10,) dtype=int8
starts               shape=(10,) dtype=float32
ends                 shape=(10,) dtype=float32
face_counts          shape=(10,) dtype=int8

FILE: /content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0003.npz
Keys: ['X', 'y', 'starts', 'ends', 'face_counts']
X                    shape=(1, 15, 224, 224, 3) dtype=uint8
y                    shape=(1,) dtype=i

In [ ]:
sample_path = train_npz_files[0]

with np.load(
    sample_path,
    allow_pickle=True
) as data:

    print("=" * 70)
    print("DETAILED NPZ INSPECTION")
    print("=" * 70)

    print("FILE:", sample_path)
    print("KEYS:", data.files)

    for key in data.files:

        arr = data[key]

        print("\n" + "-" * 60)
        print("KEY   :", key)
        print("SHAPE :", arr.shape)
        print("DTYPE :", arr.dtype)

        if arr.ndim <= 2:
            print(
                "FIRST VALUES:",
                arr[:10]
            )

DETAILED NPZ INSPECTION
FILE: /content/drive/MyDrive/deepfake_v2/final_mtcnn_windows/train/train_0001.npz
KEYS: ['X', 'y', 'starts', 'ends', 'face_counts']

------------------------------------------------------------
KEY   : X
SHAPE : (4, 15, 224, 224, 3)
DTYPE : uint8

------------------------------------------------------------
KEY   : y
SHAPE : (4,)
DTYPE : int8
FIRST VALUES: [0 0 0 0]

------------------------------------------------------------
KEY   : starts
SHAPE : (4,)
DTYPE : float32
FIRST VALUES: [92. 93. 32. 96.]

------------------------------------------------------------
KEY   : ends
SHAPE : (4,)
DTYPE : float32
FIRST VALUES: [94. 95. 34. 98.]

------------------------------------------------------------
KEY   : face_counts
SHAPE : (4,)
DTYPE : int8
FIRST VALUES: [15 15 15 15]


In [ ]:
import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/deepfake_v2"
WINDOW_ROOT = f"{BASE}/final_mtcnn_windows"

TRAIN_DIR = f"{WINDOW_ROOT}/train"
VAL_DIR   = f"{WINDOW_ROOT}/val"
TEST_DIR  = f"{WINDOW_ROOT}/test"

TRAIN_META = f"{WINDOW_ROOT}/train_window_metadata.csv"
VAL_META   = f"{WINDOW_ROOT}/val_window_metadata.csv"
TEST_META  = f"{WINDOW_ROOT}/test_window_metadata.csv"


def inspect_split(split_name, folder, metadata_path):

    metadata = pd.read_csv(metadata_path)

    files = sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.endswith(".npz")
    ])

    total_windows = 0
    real_windows = 0
    fake_windows = 0

    problems = []

    for path in files:

        try:
            with np.load(path, allow_pickle=False) as data:

                X = data["X"]
                y = data["y"]

                if len(X) != len(y):
                    problems.append(
                        f"{os.path.basename(path)}: "
                        f"X={len(X)}, y={len(y)}"
                    )
                    continue

                total_windows += len(y)
                real_windows += int(
                    np.sum(y == 0)
                )
                fake_windows += int(
                    np.sum(y == 1)
                )

        except Exception as exc:
            problems.append(
                f"{os.path.basename(path)}: {exc}"
            )

    print("=" * 70)
    print(split_name)
    print("=" * 70)

    print("NPZ files       :", len(files))
    print("NPZ windows     :", total_windows)
    print("Metadata rows   :", len(metadata))

    print()
    print("NPZ REAL        :", real_windows)
    print("NPZ FAKE        :", fake_windows)

    print()
    print("Metadata REAL   :", int((metadata["label"] == 0).sum()))
    print("Metadata FAKE   :", int((metadata["label"] == 1).sum()))

    print()

    if total_windows == len(metadata):
        print("✅ WINDOW COUNT MATCH")
    else:
        print("❌ WINDOW COUNT MISMATCH")

    if (
        real_windows
        ==
        int((metadata["label"] == 0).sum())
        and
        fake_windows
        ==
        int((metadata["label"] == 1).sum())
    ):
        print("✅ LABEL COUNTS MATCH")
    else:
        print("❌ LABEL COUNTS MISMATCH")

    if problems:
        print()
        print("Problems:")
        for problem in problems:
            print(" -", problem)

    return {
        "files": len(files),
        "windows": total_windows,
        "real": real_windows,
        "fake": fake_windows,
        "metadata": len(metadata)
    }


train_check = inspect_split(
    "TRAIN",
    TRAIN_DIR,
    TRAIN_META
)

val_check = inspect_split(
    "VALIDATION",
    VAL_DIR,
    VAL_META
)

test_check = inspect_split(
    "TEST",
    TEST_DIR,
    TEST_META
)

TRAIN
NPZ files       : 232
NPZ windows     : 1783
Metadata rows   : 1783

NPZ REAL        : 695
NPZ FAKE        : 1088

Metadata REAL   : 695
Metadata FAKE   : 1088

✅ WINDOW COUNT MATCH
✅ LABEL COUNTS MATCH
VALIDATION
NPZ files       : 50
NPZ windows     : 3391
Metadata rows   : 3391

NPZ REAL        : 2158
NPZ FAKE        : 1233

Metadata REAL   : 2158
Metadata FAKE   : 1233

✅ WINDOW COUNT MATCH
✅ LABEL COUNTS MATCH
TEST
NPZ files       : 51
NPZ windows     : 3415
Metadata rows   : 3415

NPZ REAL        : 2114
NPZ FAKE        : 1301

Metadata REAL   : 2114
Metadata FAKE   : 1301

✅ WINDOW COUNT MATCH
✅ LABEL COUNTS MATCH


In [ ]:
import os
import numpy as np

BASE = "/content/drive/MyDrive/deepfake_v2"

TRAIN_DIR = f"{BASE}/final_mtcnn_windows/train"

train_npz_files = sorted([
    os.path.join(TRAIN_DIR, f)
    for f in os.listdir(TRAIN_DIR)
    if f.endswith(".npz")
])

X_original_list = []
y_original_list = []

print("=" * 70)
print("LOADING ORIGINAL TRAINING WINDOWS")
print("=" * 70)

for i, path in enumerate(train_npz_files, start=1):

    with np.load(path, allow_pickle=False) as data:
        X_original_list.append(
            data["X"]
        )

        y_original_list.append(
            data["y"].astype(np.int64)
        )

    if i % 25 == 0 or i == len(train_npz_files):
        print(
            f"Loaded {i}/{len(train_npz_files)} shards"
        )

X_original = np.concatenate(
    X_original_list,
    axis=0
)

y_original = np.concatenate(
    y_original_list,
    axis=0
)

del X_original_list
del y_original_list

print()
print("X_original:", X_original.shape)
print("y_original:", y_original.shape)

print("REAL:", np.sum(y_original == 0))
print("FAKE:", np.sum(y_original == 1))

assert len(X_original) == 1783
assert np.sum(y_original == 0) == 695
assert np.sum(y_original == 1) == 1088

print()
print("✅ ORIGINAL TRAINING WINDOWS LOADED")

LOADING ORIGINAL TRAINING WINDOWS
Loaded 25/232 shards
Loaded 50/232 shards
Loaded 75/232 shards
Loaded 100/232 shards
Loaded 125/232 shards
Loaded 150/232 shards
Loaded 175/232 shards
Loaded 200/232 shards
Loaded 225/232 shards
Loaded 232/232 shards

X_original: (1783, 15, 224, 224, 3)
y_original: (1783,)
REAL: 695
FAKE: 1088

✅ ORIGINAL TRAINING WINDOWS LOADED


In [ ]:
import pandas as pd

HARD_TRAIN_META = (
    f"{BASE}/hard_negative_windows_v3/"
    "train_metadata.csv"
)

hard_train_df = pd.read_csv(
    HARD_TRAIN_META
)

print("Rows:", len(hard_train_df))

print(
    hard_train_df[
        "label"
    ].value_counts()
)

print(
    hard_train_df[
        "hard_negative_split"
    ].value_counts()
)

Rows: 544
label
0    544
Name: count, dtype: int64
hard_negative_split
train    544
Name: count, dtype: int64


In [ ]:
sample_hard_path = str(
    hard_train_df.iloc[0]["window_path"]
)

print("Sample:")
print(sample_hard_path)

with np.load(
    sample_hard_path,
    allow_pickle=False
) as data:

    print("Keys:", data.files)

    for key in data.files:
        print(
            key,
            data[key].shape,
            data[key].dtype
        )

Sample:
/content/drive/MyDrive/deepfake_v2/hard_negative_windows_v3/train/hardreal_001_w0000.npz
Keys: ['X', 'y']
X (15, 224, 224, 3) uint8
y () int64


In [ ]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

# Original window indices
original_real_indices = np.where(
    y_original == 0
)[0]

original_fake_indices = np.where(
    y_original == 1
)[0]

print("Available original REAL:", len(original_real_indices))
print("Available original FAKE:", len(original_fake_indices))

# Select 544 of the 695 original REAL windows.
selected_original_real = rng.choice(
    original_real_indices,
    size=544,
    replace=False
)

# Use all original FAKE windows.
selected_original_fake = original_fake_indices.copy()

print()
print("Selected original REAL :", len(selected_original_real))
print("Selected original FAKE :", len(selected_original_fake))
print("Hard-negative REAL     :", len(hard_train_df))

print()
print("Total REAL:",
      len(selected_original_real) + len(hard_train_df))

print("Total FAKE:",
      len(selected_original_fake))

Available original REAL: 695
Available original FAKE: 1088

Selected original REAL : 544
Selected original FAKE : 1088
Hard-negative REAL     : 544

Total REAL: 1088
Total FAKE: 1088


In [ ]:
print("X_original:", X_original.shape, X_original.dtype)
print("y_original:", y_original.shape, y_original.dtype)

print(
    "Memory:",
    round(X_original.nbytes / (1024**3), 2),
    "GB"
)

X_original: (1783, 15, 224, 224, 3) uint8
y_original: (1783,) int64
Memory: 3.75 GB


In [ ]:
del X_original
import gc
gc.collect()

print("X_original removed from RAM.")

X_original removed from RAM.


In [ ]:
import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/deepfake_v2"

TRAIN_DIR = f"{BASE}/final_mtcnn_windows/train"

train_npz_files = sorted([
    os.path.join(TRAIN_DIR, f)
    for f in os.listdir(TRAIN_DIR)
    if f.endswith(".npz")
])

original_index_records = []

global_index = 0

for shard_path in train_npz_files:

    with np.load(
        shard_path,
        allow_pickle=False
    ) as data:

        y = data["y"].astype(np.int64)

        for local_index, label in enumerate(y):

            original_index_records.append({
                "global_index": global_index,
                "shard_path": shard_path,
                "local_index": local_index,
                "label": int(label)
            })

            global_index += 1


original_index_df = pd.DataFrame(
    original_index_records
)

print("=" * 70)
print("ORIGINAL SHARD INDEX")
print("=" * 70)

print("Indexed windows:", len(original_index_df))
print()

print(
    original_index_df["label"].value_counts()
)

assert len(original_index_df) == 1783
assert (original_index_df["label"] == 0).sum() == 695
assert (original_index_df["label"] == 1).sum() == 1088

print()
print("✅ SHARD INDEX CREATED")

ORIGINAL SHARD INDEX
Indexed windows: 1783

label
1    1088
0     695
Name: count, dtype: int64

✅ SHARD INDEX CREATED


In [ ]:
SEED = 42

original_real_df = original_index_df[
    original_index_df["label"] == 0
].sample(
    n=544,
    random_state=SEED
)

original_fake_df = original_index_df[
    original_index_df["label"] == 1
].copy()

print("Original REAL selected :", len(original_real_df))
print("Original FAKE selected :", len(original_fake_df))
print("Hard-negative REAL     :", len(hard_train_df))

print()
print(
    "Total REAL:",
    len(original_real_df)
    +
    len(hard_train_df)
)

print(
    "Total FAKE:",
    len(original_fake_df)
)

Original REAL selected : 544
Original FAKE selected : 1088
Hard-negative REAL     : 544

Total REAL: 1088
Total FAKE: 1088


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from keras.saving import register_keras_serializable


BASE = "/content/drive/MyDrive/deepfake_v2"

MODEL_DIR = f"{BASE}/deepfake_models"

CURRENT_MODEL = (
    f"{MODEL_DIR}/condition_c_v3_best.keras"
)

HARDNEG_MODEL = (
    f"{MODEL_DIR}/condition_c_v3_hardneg_best.keras"
)


@register_keras_serializable()
class ChannelAttention(tf.keras.layers.Layer):

    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):

        channels = int(input_shape[-1])

        hidden = max(
            channels // self.ratio,
            1
        )

        self.fc1 = tf.keras.layers.Dense(
            hidden,
            activation="relu"
        )

        self.fc2 = tf.keras.layers.Dense(
            channels
        )

        super().build(input_shape)

    def call(self, x):

        avg = tf.reduce_mean(
            x,
            axis=[1, 2],
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=[1, 2],
            keepdims=True
        )

        attention = tf.nn.sigmoid(
            self.fc2(self.fc1(avg))
            +
            self.fc2(self.fc1(mx))
        )

        return x * attention

    def get_config(self):

        config = super().get_config()

        config.update({
            "ratio": self.ratio
        })

        return config


@register_keras_serializable()
class SpatialAttention(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):

        self.conv = tf.keras.layers.Conv2D(
            1,
            kernel_size=7,
            padding="same",
            activation="sigmoid"
        )

        super().build(input_shape)

    def call(self, x):

        avg = tf.reduce_mean(
            x,
            axis=-1,
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=-1,
            keepdims=True
        )

        concat = tf.concat(
            [avg, mx],
            axis=-1
        )

        return x * self.conv(concat)


model = tf.keras.models.load_model(
    CURRENT_MODEL,
    custom_objects={
        "ChannelAttention": ChannelAttention,
        "SpatialAttention": SpatialAttention
    },
    compile=False,
    safe_mode=False
)

print("MODEL LOADED")
print("Input :", model.input_shape)
print("Output:", model.output_shape)

MODEL LOADED
Input : (None, 15, 224, 224, 3)
Output: (None, 2)


In [ ]:
td_layer = model.get_layer(
    "td_spatial"
)

spatial_model = td_layer.layer

backbone = None

for layer in spatial_model.layers:

    if isinstance(
        layer,
        tf.keras.Model
    ):

        if "mobilenet" in layer.name.lower():
            backbone = layer
            break


if backbone is None:
    raise RuntimeError(
        "MobileNetV2 backbone not found."
    )


backbone.trainable = True

for layer in backbone.layers:
    layer.trainable = False


for layer in backbone.layers[-20:]:

    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True


print(
    "Trainable MobileNetV2 layers:",
    sum(
        layer.trainable
        for layer in backbone.layers
    )
)

Trainable MobileNetV2 layers: 13


In [ ]:
BATCH_SIZE = 4
SEED = 42


original_real_records = (
    original_real_df[
        [
            "shard_path",
            "local_index",
            "label"
        ]
    ]
    .to_dict(
        orient="records"
    )
)

original_fake_records = (
    original_fake_df[
        [
            "shard_path",
            "local_index",
            "label"
        ]
    ]
    .to_dict(
        orient="records"
    )
)

hard_records = []

for _, row in hard_train_df.iterrows():

    hard_records.append({
        "window_path":
            str(
                row["window_path"]
            ),

        "label":
            0
    })


print("Original REAL:", len(original_real_records))
print("Original FAKE:", len(original_fake_records))
print("Hard REAL    :", len(hard_records))

Original REAL: 544
Original FAKE: 1088
Hard REAL    : 544


In [ ]:
def hard_negative_finetune_generator():

    samples = []

    for record in original_real_records:

        samples.append(
            (
                "original",
                record
            )
        )

    for record in original_fake_records:

        samples.append(
            (
                "original",
                record
            )
        )

    for record in hard_records:

        samples.append(
            (
                "hard",
                record
            )
        )


    rng = np.random.default_rng()

    order = rng.permutation(
        len(samples)
    )


    for index in order:

        source, record = samples[index]


        if source == "original":

            with np.load(
                record["shard_path"],
                allow_pickle=False
            ) as data:

                X = data["X"][
                    int(
                        record["local_index"]
                    )
                ]

                y = int(
                    record["label"]
                )


        else:

            with np.load(
                record["window_path"],
                allow_pickle=False
            ) as data:

                X = data["X"]

                y = 0


        # EXACT NORMALIZATION USED IN VERSION5
        X = (
            X.astype(
                np.float32
            )
            /
            255.0
        )

        yield (
            X,
            np.int32(y)
        )

In [ ]:
OUTPUT_SIGNATURE = (
    tf.TensorSpec(
        shape=(
            15,
            224,
            224,
            3
        ),
        dtype=tf.float32
    ),

    tf.TensorSpec(
        shape=(),
        dtype=tf.int32
    )
)


finetune_ds = tf.data.Dataset.from_generator(
    hard_negative_finetune_generator,
    output_signature=OUTPUT_SIGNATURE
)


finetune_ds = (
    finetune_ds
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)


print(finetune_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 15, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


In [ ]:
for X_batch, y_batch in finetune_ds.take(1):

    print("=" * 70)
    print("FINE-TUNING BATCH CHECK")
    print("=" * 70)

    print(
        "X:",
        X_batch.shape,
        X_batch.dtype
    )

    print(
        "y:",
        y_batch.shape,
        y_batch.dtype
    )

    print(
        "Pixel min:",
        float(
            tf.reduce_min(
                X_batch
            )
        )
    )

    print(
        "Pixel max:",
        float(
            tf.reduce_max(
                X_batch
            )
        )
    )

    print(
        "Labels:",
        y_batch.numpy()
    )

FINE-TUNING BATCH CHECK
X: (4, 15, 224, 224, 3) <dtype: 'float32'>
y: (4,) <dtype: 'int32'>
Pixel min: 0.0
Pixel max: 1.0
Labels: [0 1 0 1]


In [ ]:
import glob

VAL_DIR = (
    f"{BASE}/final_mtcnn_windows/val"
)

VAL_META = (
    f"{BASE}/final_mtcnn_windows/"
    "val_window_metadata.csv"
)

val_window_df = pd.read_csv(
    VAL_META
)

val_files = sorted(
    glob.glob(
        f"{VAL_DIR}/*.npz"
    )
)


def original_npz_generator(
    file_list
):

    for file_path in file_list:

        with np.load(
            file_path,
            allow_pickle=False
        ) as data:

            X = data["X"]
            y = data["y"]

            for i in range(
                len(y)
            ):

                yield (
                    X[i].astype(
                        np.float32
                    )
                    /
                    255.0,

                    np.int32(
                        y[i]
                    )
                )


val_ds = tf.data.Dataset.from_generator(
    lambda:
        original_npz_generator(
            val_files
        ),
    output_signature=
        OUTPUT_SIGNATURE
)


val_ds = (
    val_ds
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)


print(
    "Validation windows:",
    len(
        val_window_df
    )
)

Validation windows: 3391


In [ ]:
HARDNEG_LR = 1e-6

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=HARDNEG_LR
    ),

    loss=
        "sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)

In [ ]:
callbacks_hardneg = [

    tf.keras.callbacks.ModelCheckpoint(
        HARDNEG_MODEL,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]


print(
    "Saving hard-negative model to:"
)

print(
    HARDNEG_MODEL
)

Saving hard-negative model to:
/content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_hardneg_best.keras


In [ ]:
import math

FINETUNE_WINDOWS = (
    1088
    +
    1088
)

STEPS_PER_EPOCH = math.ceil(
    FINETUNE_WINDOWS
    /
    BATCH_SIZE
)

VALIDATION_STEPS = math.ceil(
    len(
        val_window_df
    )
    /
    BATCH_SIZE
)

print(
    "Fine-tune windows :",
    FINETUNE_WINDOWS
)

print(
    "Steps/epoch       :",
    STEPS_PER_EPOCH
)

print(
    "Validation steps  :",
    VALIDATION_STEPS
)

Fine-tune windows : 2176
Steps/epoch       : 544
Validation steps  : 848


In [ ]:
history_hardneg = model.fit(

    finetune_ds,

    validation_data=
        val_ds,

    epochs=5,

    callbacks=
        callbacks_hardneg,

    verbose=1
)

Epoch 1/5
    544/Unknown 926s 2s/step - accuracy: 0.8508 - loss: 0.7045

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: val_loss improved from None to 0.32449, saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_hardneg_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_hardneg_best.keras
544/544 ━━━━━━━━━━━━━━━━━━━━ 1072s 2s/step - accuracy: 0.8869 - loss: 0.4209 - val_accuracy: 0.8838 - val_loss: 0.3245 - learning_rate: 1.0000e-06
Epoch 2/5
544/544 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.9346 - loss: 0.1864
Epoch 2: val_loss did not improve from 0.32449

Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999987376214e-07.
544/544 ━━━━━━━━━━━━━━━━━━━━ 296s 545ms/step - accuracy: 0.9412 - loss: 0.1697 - val_accuracy: 0.8773 - val_loss: 0.3279 - learning_rate: 1.0000e-06
Epoch 3/5
544/544 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step - accuracy: 0.9576 - loss: 0.1166
Epoch 3: val_loss improved from 0.32449 to 0.30599, saving model to /content/drive/MyDrive/deepfake_v2/deepfake_models/condition_c_v3_h

In [ ]:
HISTORY_DIR = (
    f"{BASE}/final_training_results"
)

os.makedirs(
    HISTORY_DIR,
    exist_ok=True
)


history_hardneg_df = pd.DataFrame(
    history_hardneg.history
)


history_hardneg_df.to_csv(
    f"{HISTORY_DIR}/"
    "condition_c_v3_hardneg_history.csv",
    index=False
)


display(
    history_hardneg_df
)

print(
    "Hard-negative history saved."
)

,accuracy,loss,val_accuracy,val_loss,learning_rate
0,0.886949,0.420907,0.883810,0.324491,1.000000e-06
1,0.941176,0.169694,0.877322,0.327864,1.000000e-06
2,0.960478,0.120902,0.885285,0.305993,5.000000e-07
3,0.967371,0.096006,0.887349,0.296579,5.000000e-07
4,0.976103,0.076752,0.893542,0.295158,5.000000e-07


Hard-negative history saved.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

HARDNEG_MODEL = (
    f"{BASE}/deepfake_models/"
    "condition_c_v3_hardneg_best.keras"
)

hardneg_model = tf.keras.models.load_model(
    HARDNEG_MODEL,
    custom_objects={
        "ChannelAttention": ChannelAttention,
        "SpatialAttention": SpatialAttention
    },
    compile=False,
    safe_mode=False
)

print("Hard-negative model loaded.")
print("Input :", hardneg_model.input_shape)
print("Output:", hardneg_model.output_shape)

Hard-negative model loaded.
Input : (None, 15, 224, 224, 3)
Output: (None, 2)


In [ ]:
val_probs = hardneg_model.predict(
    val_ds,
    verbose=1
)

y_val_true = (
    val_window_df["label"]
    .astype(int)
    .values
)

fake_probs_val = val_probs[:, 1]

VAL_THRESHOLD = 0.65

y_val_pred = (
    fake_probs_val
    >=
    VAL_THRESHOLD
).astype(int)

848/848 ━━━━━━━━━━━━━━━━━━━━ 157s 167ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
val_accuracy = accuracy_score(
    y_val_true,
    y_val_pred
)

val_precision = precision_score(
    y_val_true,
    y_val_pred,
    pos_label=1,
    zero_division=0
)

val_recall = recall_score(
    y_val_true,
    y_val_pred,
    pos_label=1,
    zero_division=0
)

val_f1 = f1_score(
    y_val_true,
    y_val_pred,
    pos_label=1,
    zero_division=0
)

val_auc = roc_auc_score(
    y_val_true,
    fake_probs_val
)

val_cm = confusion_matrix(
    y_val_true,
    y_val_pred
)

print("=" * 70)
print("HARDNEG MODEL — ORIGINAL VALIDATION")
print("=" * 70)

print(f"Threshold : {VAL_THRESHOLD:.2f}")
print(f"Accuracy  : {val_accuracy:.4f}")
print(f"Precision : {val_precision:.4f}")
print(f"Recall    : {val_recall:.4f}")
print(f"F1-score  : {val_f1:.4f}")
print(f"ROC-AUC   : {val_auc:.4f}")

print()
print("Confusion Matrix:")
print(val_cm)

print()
print(
    classification_report(
        y_val_true,
        y_val_pred,
        target_names=[
            "REAL",
            "FAKE"
        ],
        digits=4
    )
)

HARDNEG MODEL — ORIGINAL VALIDATION
Threshold : 0.65
Accuracy  : 0.8986
Precision : 0.9082
Recall    : 0.8021
F1-score  : 0.8519
ROC-AUC   : 0.9541

Confusion Matrix:
[[2058  100]
 [ 244  989]]

              precision    recall  f1-score   support

        REAL     0.8940    0.9537    0.9229      2158
        FAKE     0.9082    0.8021    0.8519      1233

    accuracy                         0.8986      3391
   macro avg     0.9011    0.8779    0.8874      3391
weighted avg     0.8992    0.8986    0.8970      3391



In [ ]:
TEST_DIR = (
    f"{BASE}/final_mtcnn_windows/test"
)

TEST_META = (
    f"{BASE}/final_mtcnn_windows/"
    "test_window_metadata.csv"
)

test_window_df = pd.read_csv(
    TEST_META
)

test_files = sorted(
    glob.glob(
        f"{TEST_DIR}/*.npz"
    )
)

test_ds = tf.data.Dataset.from_generator(
    lambda:
        original_npz_generator(
            test_files
        ),
    output_signature=
        OUTPUT_SIGNATURE
)

test_ds = (
    test_ds
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)

print(
    "Test windows:",
    len(
        test_window_df
    )
)

Test windows: 3415


In [ ]:
test_probs = hardneg_model.predict(
    test_ds,
    verbose=1
)

y_test_true = (
    test_window_df["label"]
    .astype(int)
    .values
)

fake_probs_test = (
    test_probs[:, 1]
)

TEST_THRESHOLD = 0.65

y_test_pred = (
    fake_probs_test
    >=
    TEST_THRESHOLD
).astype(int)

854/854 ━━━━━━━━━━━━━━━━━━━━ 127s 148ms/step


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
test_accuracy = accuracy_score(
    y_test_true,
    y_test_pred
)

test_precision = precision_score(
    y_test_true,
    y_test_pred,
    pos_label=1,
    zero_division=0
)

test_recall = recall_score(
    y_test_true,
    y_test_pred,
    pos_label=1,
    zero_division=0
)

test_f1 = f1_score(
    y_test_true,
    y_test_pred,
    pos_label=1,
    zero_division=0
)

test_auc = roc_auc_score(
    y_test_true,
    fake_probs_test
)

test_specificity = recall_score(
    y_test_true,
    y_test_pred,
    pos_label=0,
    zero_division=0
)

test_cm = confusion_matrix(
    y_test_true,
    y_test_pred
)

print("=" * 70)
print("HARDNEG MODEL — ORIGINAL TEST")
print("=" * 70)

print(f"Threshold   : {TEST_THRESHOLD:.2f}")
print(f"Accuracy    : {test_accuracy:.4f}")
print(f"Precision   : {test_precision:.4f}")
print(f"Recall      : {test_recall:.4f}")
print(f"F1-score    : {test_f1:.4f}")
print(f"ROC-AUC     : {test_auc:.4f}")
print(f"Specificity : {test_specificity:.4f}")

print()
print("Confusion Matrix:")
print(test_cm)

print()
print(
    classification_report(
        y_test_true,
        y_test_pred,
        target_names=[
            "REAL",
            "FAKE"
        ],
        digits=4
    )
)

HARDNEG MODEL — ORIGINAL TEST
Threshold   : 0.65
Accuracy    : 0.8515
Precision   : 0.8161
Recall      : 0.7879
F1-score    : 0.8017
ROC-AUC     : 0.9220
Specificity : 0.8907

Confusion Matrix:
[[1883  231]
 [ 276 1025]]

              precision    recall  f1-score   support

        REAL     0.8722    0.8907    0.8813      2114
        FAKE     0.8161    0.7879    0.8017      1301

    accuracy                         0.8515      3415
   macro avg     0.8441    0.8393    0.8415      3415
weighted avg     0.8508    0.8515    0.8510      3415



In [ ]:
HARD_HOLDOUT_META = (
    f"{BASE}/hard_negative_windows_v3/"
    "holdout_metadata.csv"
)

hard_holdout_df = pd.read_csv(
    HARD_HOLDOUT_META
)

print(
    "Holdout windows:",
    len(
        hard_holdout_df
    )
)

print(
    "Holdout videos:",
    hard_holdout_df[
        "video_id"
    ].nunique()
)

Holdout windows: 130
Holdout videos: 5


In [ ]:
def hard_holdout_generator():

    for _, row in hard_holdout_df.iterrows():

        path = str(
            row["window_path"]
        )

        with np.load(
            path,
            allow_pickle=False
        ) as data:

            X = data["X"]

        X = (
            X.astype(
                np.float32
            )
            /
            255.0
        )

        yield (
            X,
            np.int32(0)
        )


hard_holdout_ds = (
    tf.data.Dataset.from_generator(
        hard_holdout_generator,
        output_signature=
            OUTPUT_SIGNATURE
    )
    .batch(
        BATCH_SIZE
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)

In [ ]:
holdout_probs = hardneg_model.predict(
    hard_holdout_ds,
    verbose=1
)

hard_holdout_df[
    "fake_probability"
] = holdout_probs[:, 1]

hard_holdout_df[
    "window_prediction"
] = (
    hard_holdout_df[
        "fake_probability"
    ]
    >=
    0.65
).astype(int)

33/33 ━━━━━━━━━━━━━━━━━━━━ 148s 4s/step


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
holdout_video_summary = (
    hard_holdout_df
    .groupby("video_id")
    .agg(
        windows=(
            "window_prediction",
            "size"
        ),
        fake_windows=(
            "window_prediction",
            "sum"
        ),
        mean_fake_probability=(
            "fake_probability",
            "mean"
        ),
        max_fake_probability=(
            "fake_probability",
            "max"
        )
    )
    .reset_index()
)

holdout_video_summary[
    "fake_percentage"
] = (
    holdout_video_summary[
        "fake_windows"
    ]
    /
    holdout_video_summary[
        "windows"
    ]
    *
    100.0
)

display(
    holdout_video_summary
)

,video_id,windows,fake_windows,mean_fake_probability,max_fake_probability,fake_percentage
0,hardreal_005,30,0,0.021658,0.094421,0.000000
1,hardreal_006,29,4,0.166130,0.980070,13.793103
2,hardreal_010,26,1,0.070910,0.977794,3.846154
3,hardreal_015,23,7,0.363652,0.999392,30.434783
4,hardreal_020,22,3,0.281674,0.965644,13.636364


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

threshold_rows = []

for threshold in np.arange(
    0.30,
    0.81,
    0.05
):

    preds = (
        fake_probs_val
        >=
        threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_val_true,
        preds
    )

    precision_fake = precision_score(
        y_val_true,
        preds,
        pos_label=1,
        zero_division=0
    )

    recall_fake = recall_score(
        y_val_true,
        preds,
        pos_label=1,
        zero_division=0
    )

    f1_fake = f1_score(
        y_val_true,
        preds,
        pos_label=1,
        zero_division=0
    )

    specificity_real = recall_score(
        y_val_true,
        preds,
        pos_label=0,
        zero_division=0
    )

    threshold_rows.append({
        "threshold":
            round(
                float(threshold),
                2
            ),

        "accuracy":
            accuracy,

        "precision_fake":
            precision_fake,

        "recall_fake":
            recall_fake,

        "f1_fake":
            f1_fake,

        "specificity_real":
            specificity_real
    })


threshold_df = pd.DataFrame(
    threshold_rows
)

print("=" * 90)
print("HARDNEG MODEL — VALIDATION THRESHOLD SWEEP")
print("=" * 90)

display(
    threshold_df.round(4)
)

HARDNEG MODEL — VALIDATION THRESHOLD SWEEP


,threshold,accuracy,precision_fake,recall_fake,f1_fake,specificity_real
0,0.30,0.8785,0.8126,0.8654,0.8382,0.8860
1,0.35,0.8797,0.8210,0.8556,0.8380,0.8934
2,0.40,0.8829,0.8328,0.8483,0.8405,0.9027
3,0.45,0.8879,0.8505,0.8394,0.8449,0.9157
4,0.50,0.8935,0.8707,0.8305,0.8501,0.9296
5,0.55,0.8959,0.8846,0.8208,0.8515,0.9388
6,0.60,0.8991,0.8996,0.8135,0.8543,0.9481
7,0.65,0.8986,0.9082,0.8021,0.8519,0.9537
8,0.70,0.8977,0.9148,0.7924,0.8492,0.9578
9,0.75,0.8968,0.9241,0.7802,0.8461,0.9634


In [ ]:
threshold_df[
    "balanced_score"
] = (
    threshold_df[
        "recall_fake"
    ]
    +
    threshold_df[
        "specificity_real"
    ]
) / 2.0


best_row = threshold_df.loc[
    threshold_df[
        "balanced_score"
    ].idxmax()
]


print("=" * 70)
print("BEST BALANCED VALIDATION THRESHOLD")
print("=" * 70)

print(best_row)

BEST BALANCED VALIDATION THRESHOLD
threshold           0.600000
accuracy            0.899145
precision_fake      0.899552
recall_fake         0.813463
f1_fake             0.854344
specificity_real    0.948100
balanced_score      0.880782
Name: 6, dtype: float64


In [ ]:
FINAL_THRESHOLD = 0.60

y_test_pred_060 = (
    fake_probs_test >= FINAL_THRESHOLD
).astype(int)

test_accuracy_060 = accuracy_score(
    y_test_true,
    y_test_pred_060
)

test_precision_060 = precision_score(
    y_test_true,
    y_test_pred_060,
    pos_label=1,
    zero_division=0
)

test_recall_060 = recall_score(
    y_test_true,
    y_test_pred_060,
    pos_label=1,
    zero_division=0
)

test_f1_060 = f1_score(
    y_test_true,
    y_test_pred_060,
    pos_label=1,
    zero_division=0
)

test_specificity_060 = recall_score(
    y_test_true,
    y_test_pred_060,
    pos_label=0,
    zero_division=0
)

test_auc_060 = roc_auc_score(
    y_test_true,
    fake_probs_test
)

test_cm_060 = confusion_matrix(
    y_test_true,
    y_test_pred_060
)

print("=" * 70)
print("FINAL HARDNEG MODEL — ORIGINAL TEST")
print("VALIDATION-SELECTED THRESHOLD = 0.60")
print("=" * 70)

print(f"Accuracy    : {test_accuracy_060:.4f}")
print(f"Precision   : {test_precision_060:.4f}")
print(f"Recall      : {test_recall_060:.4f}")
print(f"F1-score    : {test_f1_060:.4f}")
print(f"ROC-AUC     : {test_auc_060:.4f}")
print(f"Specificity : {test_specificity_060:.4f}")

print()
print("Confusion Matrix:")
print(test_cm_060)

print()
print(
    classification_report(
        y_test_true,
        y_test_pred_060,
        target_names=["REAL", "FAKE"],
        digits=4
    )
)

FINAL HARDNEG MODEL — ORIGINAL TEST
VALIDATION-SELECTED THRESHOLD = 0.60
Accuracy    : 0.8515
Precision   : 0.8068
Recall      : 0.8025
F1-score    : 0.8046
ROC-AUC     : 0.9220
Specificity : 0.8817

Confusion Matrix:
[[1864  250]
 [ 257 1044]]

              precision    recall  f1-score   support

        REAL     0.8788    0.8817    0.8803      2114
        FAKE     0.8068    0.8025    0.8046      1301

    accuracy                         0.8515      3415
   macro avg     0.8428    0.8421    0.8425      3415
weighted avg     0.8514    0.8515    0.8515      3415



In [ ]:
hard_holdout_df["window_prediction_060"] = (
    hard_holdout_df["fake_probability"] >= 0.60
).astype(int)

holdout_summary_060 = (
    hard_holdout_df
    .groupby("video_id")
    .agg(
        windows=(
            "window_prediction_060",
            "size"
        ),
        fake_windows=(
            "window_prediction_060",
            "sum"
        ),
        mean_fake_probability=(
            "fake_probability",
            "mean"
        ),
        max_fake_probability=(
            "fake_probability",
            "max"
        )
    )
    .reset_index()
)

holdout_summary_060["fake_percentage"] = (
    holdout_summary_060["fake_windows"]
    /
    holdout_summary_060["windows"]
    *
    100.0
)

print("=" * 80)
print("UNTOUCHED REAL HOLDOUT — THRESHOLD 0.60")
print("=" * 80)

display(
    holdout_summary_060.round(4)
)

UNTOUCHED REAL HOLDOUT — THRESHOLD 0.60


,video_id,windows,fake_windows,mean_fake_probability,max_fake_probability,fake_percentage
0,hardreal_005,30,0,0.0217,0.0944,0.0000
1,hardreal_006,29,4,0.1661,0.9801,13.7931
2,hardreal_010,26,1,0.0709,0.9778,3.8462
3,hardreal_015,23,7,0.3637,0.9994,30.4348
4,hardreal_020,22,3,0.2817,0.9656,13.6364


In [ ]:
import hashlib

p = (
    "/content/drive/MyDrive/deepfake_v2/"
    "deepfake_models/"
    "condition_c_v3_hardneg_best.keras"
)

with open(p, "rb") as f:
    print(hashlib.sha256(f.read()).hexdigest())

c03dbc13be440e265ec56416cba2fc92a3fcf7f0ec554311f0204ca83557cdc4


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import tensorflow as tf
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/deepfake_v2"

KERAS_MODEL_PATH = (
    f"{BASE}/deepfake_models/"
    "condition_c_v3_hardneg_best.keras"
)

TFLITE_DIR = (
    f"{BASE}/tflite_models"
)

os.makedirs(
    TFLITE_DIR,
    exist_ok=True
)

FP32_PATH = (
    f"{TFLITE_DIR}/"
    "condition_c_v3_hardneg_fp32.tflite"
)

FP16_PATH = (
    f"{TFLITE_DIR}/"
    "condition_c_v3_hardneg_fp16.tflite"
)

print(
    "Keras exists:",
    os.path.exists(
        KERAS_MODEL_PATH
    )
)

Keras exists: True


In [3]:
from keras.saving import register_keras_serializable


@register_keras_serializable()
class ChannelAttention(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        ratio=8,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(
        self,
        input_shape
    ):

        channels = int(
            input_shape[-1]
        )

        hidden = max(
            channels // self.ratio,
            1
        )

        self.fc1 = tf.keras.layers.Dense(
            hidden,
            activation="relu"
        )

        self.fc2 = tf.keras.layers.Dense(
            channels
        )

        super().build(
            input_shape
        )

    def call(
        self,
        x
    ):

        avg = tf.reduce_mean(
            x,
            axis=[1, 2],
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=[1, 2],
            keepdims=True
        )

        avg_out = self.fc2(
            self.fc1(
                avg
            )
        )

        max_out = self.fc2(
            self.fc1(
                mx
            )
        )

        attention = tf.nn.sigmoid(
            avg_out
            +
            max_out
        )

        return (
            x
            *
            attention
        )

    def get_config(
        self
    ):

        config = (
            super()
            .get_config()
        )

        config.update({
            "ratio":
                self.ratio
        })

        return config


@register_keras_serializable()
class SpatialAttention(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        **kwargs
    ):
        super().__init__(
            **kwargs
        )

    def build(
        self,
        input_shape
    ):

        self.conv = (
            tf.keras.layers.Conv2D(
                1,
                kernel_size=7,
                padding="same",
                activation="sigmoid"
            )
        )

        super().build(
            input_shape
        )

    def call(
        self,
        x
    ):

        avg = tf.reduce_mean(
            x,
            axis=-1,
            keepdims=True
        )

        mx = tf.reduce_max(
            x,
            axis=-1,
            keepdims=True
        )

        concat = tf.concat(
            [
                avg,
                mx
            ],
            axis=-1
        )

        attention = self.conv(
            concat
        )

        return (
            x
            *
            attention
        )


hardneg_model = (
    tf.keras.models.load_model(
        KERAS_MODEL_PATH,
        custom_objects={
            "ChannelAttention":
                ChannelAttention,

            "SpatialAttention":
                SpatialAttention
        },
        compile=False,
        safe_mode=False
    )
)

print(
    hardneg_model.input_shape
)

print(
    hardneg_model.output_shape
)

(None, 15, 224, 224, 3)
(None, 2)


In [4]:
converter = (
    tf.lite.TFLiteConverter
    .from_keras_model(
        hardneg_model
    )
)

tflite_fp32 = (
    converter.convert()
)

with open(
    FP32_PATH,
    "wb"
) as f:

    f.write(
        tflite_fp32
    )

print(
    "FP32 TFLite saved:"
)

print(
    FP32_PATH
)

print(
    "FP32 size:",
    round(
        os.path.getsize(
            FP32_PATH
        )
        /
        (1024 ** 2),
        2
    ),
    "MB"
)

Saved artifact at '/tmp/tmpr4t8pqzy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 15, 224, 224, 3), dtype=tf.float32, name='sequence_input')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  135591429081040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429081616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429081424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429082000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429080848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429082192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429082960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429082768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429083152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135591429081232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13559142

ConverterError: Could not translate MLIR to FlatBuffer.<unknown>:0: error: loc(callsite(fused["CudnnRNNV3:", "Condition_C_v3_1/bigru_1_1/backward_gru_1/CudnnRNNV3@__inference_function_30073"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_31220"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]))): 'tf.CudnnRNNV3' op is neither a custom op nor a flex op
<unknown>:0: note: loc(callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_31220"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): called from
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): called from
<unknown>:0: note: loc(callsite(fused["CudnnRNNV3:", "Condition_C_v3_1/bigru_1_1/backward_gru_1/CudnnRNNV3@__inference_function_30073"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_31220"] at fused["Stateful

In [5]:
print("=" * 100)
print("MODEL SUMMARY")
print("=" * 100)

hardneg_model.summary()

print("\n" + "=" * 100)
print("LAYER CONFIGURATION")
print("=" * 100)

for i, layer in enumerate(hardneg_model.layers):
    print(f"\n[{i}] {layer.name}")
    print("Class:", layer.__class__.__name__)

    try:
        print("Input shape :", layer.input.shape)
    except:
        pass

    try:
        print("Output shape:", layer.output.shape)
    except:
        pass

    if isinstance(layer, tf.keras.layers.Bidirectional):
        print("\n>>> BIDIRECTIONAL RNN FOUND")
        print("Forward config:")
        print(layer.forward_layer.get_config())

        print("\nBackward config:")
        print(layer.backward_layer.get_config())

MODEL SUMMARY


Model: "Condition_C_v3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequence_input (InputLayer)     │ (None, 15, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_spatial (TimeDistributed)    │ (None, 15, 256)        │     2,997,059 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bigru_1 (Bidirectional)         │ (None, 15, 128)        │       123,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bigru_2 (Bidirectional)         │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,156,101 (12.04 MB)

 Trainable params: 2,092,997 (7.98 MB)

 Non-trainable params: 1,063,104 (4.06 MB)


LAYER CONFIGURATION

[0] sequence_input
Class: InputLayer
Output shape: (None, 15, 224, 224, 3)

[1] td_spatial
Class: TimeDistributed
Input shape : (None, 15, 224, 224, 3)
Output shape: (None, 15, 256)

[2] bigru_1
Class: Bidirectional
Input shape : (None, 15, 256)
Output shape: (None, 15, 128)

>>> BIDIRECTIONAL RNN FOUND
Forward config:
{'name': 'forward_gru', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'return_sequences': True, 'return_state': False, 'go_backwards': False, 'stateful': False, 'unroll': False, 'zero_output_for_mask': True, 'units': 64, 'activation': 'tanh', 'recurrent_activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'recurrent_initializer': {'module': 'keras.initializers', 'class_name': 'Orthogonal', 'config': {'seed': None, 'gain': 1.0}, 'registere

In [6]:
import tensorflow as tf
import inspect
import numpy as np
import os

print("TensorFlow:", tf.__version__)


# ============================================================
# HELPER — CREATE NON-CUDNN GRU FROM ORIGINAL CONFIG
# ============================================================

def make_cpu_gru(
    old_gru,
    new_name
):
    config = old_gru.get_config()

    config["name"] = new_name

    # Remove fields that Keras reconstructs itself if necessary
    config.pop(
        "trainable",
        None
    )

    # Force non-cuDNN implementation when supported by
    # the installed Keras version.
    gru_signature = inspect.signature(
        tf.keras.layers.GRU.__init__
    )

    if "use_cudnn" in gru_signature.parameters:
        config[
            "use_cudnn"
        ] = False

    return tf.keras.layers.GRU.from_config(
        config
    )


# ============================================================
# ORIGINAL LAYERS
# ============================================================

old_td = hardneg_model.get_layer(
    "td_spatial"
)

old_bi1 = hardneg_model.get_layer(
    "bigru_1"
)

old_bi2 = hardneg_model.get_layer(
    "bigru_2"
)

old_dense = hardneg_model.get_layer(
    "dense_3"
)

old_dropout = hardneg_model.get_layer(
    "dropout"
)

old_output = hardneg_model.get_layer(
    "output"
)


# ============================================================
# NEW INPUT
# ============================================================

inputs = tf.keras.Input(
    shape=(
        15,
        224,
        224,
        3
    ),
    dtype=tf.float32,
    name="sequence_input"
)


# ============================================================
# SAME SPATIAL FEATURE EXTRACTOR
# ============================================================

x = old_td(
    inputs,
    training=False
)


# ============================================================
# REBUILD BiGRU 1 WITHOUT cuDNN
# ============================================================

forward_1 = make_cpu_gru(
    old_bi1.forward_layer,
    "forward_gru_cpu"
)

backward_1 = make_cpu_gru(
    old_bi1.backward_layer,
    "backward_gru_cpu"
)

new_bi1 = tf.keras.layers.Bidirectional(
    forward_1,
    backward_layer=backward_1,
    merge_mode=old_bi1.merge_mode,
    name="bigru_1_cpu"
)

x = new_bi1(
    x
)


# ============================================================
# REBUILD BiGRU 2 WITHOUT cuDNN
# ============================================================

forward_2 = make_cpu_gru(
    old_bi2.forward_layer,
    "forward_gru_2_cpu"
)

backward_2 = make_cpu_gru(
    old_bi2.backward_layer,
    "backward_gru_2_cpu"
)

new_bi2 = tf.keras.layers.Bidirectional(
    forward_2,
    backward_layer=backward_2,
    merge_mode=old_bi2.merge_mode,
    name="bigru_2_cpu"
)

x = new_bi2(
    x
)


# ============================================================
# COPY ORIGINAL CLASSIFICATION HEAD CONFIG
# ============================================================

dense_config = (
    old_dense.get_config()
)

dense_config["name"] = (
    "dense_3_cpu"
)

new_dense = (
    tf.keras.layers.Dense.from_config(
        dense_config
    )
)

x = new_dense(
    x
)


dropout_config = (
    old_dropout.get_config()
)

dropout_config["name"] = (
    "dropout_cpu"
)

new_dropout = (
    tf.keras.layers.Dropout.from_config(
        dropout_config
    )
)

x = new_dropout(
    x,
    training=False
)


output_config = (
    old_output.get_config()
)

output_config["name"] = (
    "output"
)

new_output = (
    tf.keras.layers.Dense.from_config(
        output_config
    )
)

outputs = new_output(
    x
)


# ============================================================
# FINAL INFERENCE MODEL
# ============================================================

tflite_ready_model = (
    tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="Condition_C_v3_TFLiteReady"
    )
)


print()
print("=" * 70)
print("TFLITE-READY MODEL CREATED")
print("=" * 70)

tflite_ready_model.summary()

TensorFlow: 2.20.0

TFLITE-READY MODEL CREATED


Model: "Condition_C_v3_TFLiteReady"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequence_input (InputLayer)     │ (None, 15, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ td_spatial (TimeDistributed)    │ (None, 15, 256)        │     2,997,059 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bigru_1_cpu (Bidirectional)     │ (None, 15, 128)        │       123,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bigru_2_cpu (Bidirectional)     │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3_cpu (Dense)             │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_cpu (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,156,101 (12.04 MB)

 Trainable params: 2,092,997 (7.98 MB)

 Non-trainable params: 1,063,104 (4.06 MB)

In [7]:
import glob
import numpy as np

BASE = "/content/drive/MyDrive/deepfake_v2"

TEST_DIR = (
    f"{BASE}/final_mtcnn_windows/test"
)

test_files = sorted(
    glob.glob(
        f"{TEST_DIR}/*.npz"
    )
)

sample_file = test_files[0]

with np.load(
    sample_file,
    allow_pickle=False
) as data:

    X_sample = (
        data["X"][0]
        .astype(np.float32)
        /
        255.0
    )

    y_sample = int(
        data["y"][0]
    )

X_sample = np.expand_dims(
    X_sample,
    axis=0
)

old_prediction = (
    hardneg_model(
        X_sample,
        training=False
    )
    .numpy()
)

new_prediction = (
    tflite_ready_model(
        X_sample,
        training=False
    )
    .numpy()
)

difference = np.abs(
    old_prediction
    -
    new_prediction
)

print("=" * 70)
print("ORIGINAL vs TFLITE-READY MODEL")
print("=" * 70)

print("True label:")
print(y_sample)

print()

print("Original prediction:")
print(old_prediction)

print()

print("TFLite-ready prediction:")
print(new_prediction)

print()

print("Absolute difference:")
print(difference)

print()

print(
    "Maximum difference:",
    float(
        difference.max()
    )
)

ORIGINAL vs TFLITE-READY MODEL
True label:
1

Original prediction:
[[0.00624234 0.9937577 ]]

TFLite-ready prediction:
[[0.60241026 0.39758974]]

Absolute difference:
[[0.5961679 0.596168 ]]

Maximum difference: 0.5961679816246033
